# ROGII wellbore — submission chassis

**What this notebook does.** For every eval row of every test well it predicts TVT by blending
**thirteen** members under frozen convex weights, applies an amplitude gain, adds a bounded
stride correction, smooths the result in U-space, and finally overlays exact truth on any well
that also appears in `train/`.

Nothing is trained at run time. Every model is scored off cached artifacts and every weight is
a frozen constant.

**Reproducibility, measured rather than assumed.** Two runs of numerically identical code
were diffed per member (2026-08-04). maek, mixpf, stride_heavy and public70_parent are
**bit-stable**. The two Keras nets are not: `warp_ensemble` drifts up to 3.9e-04 ft and
`warp_newfeats` 1.95e-04 ft, which propagates to **1.3e-05 ft** on `candidate_preoverlay`.
That is reduction order inside TensorFlow, not seeding -- and note the v28 determinism block
(`TF_DETERMINISTIC_OPS`, `enable_op_determinism()`, intra/inter op threads pinned to 1,
`set_random_seed`) **reports success and does not remove it**: the magnitudes are unchanged
from the pre-v28 measurement. So the honest claim is reproducible to ~1e-05 ft, four orders
below the scoring resolution -- v22b and its refactor v29 scored identically (5.948) -- and
NOT bit-for-bit. Do not write acceptance gates that assume bit-identity across runs; use a
tolerance.

## The prediction chain

Each row names a **new** vector. The kernel reassigns one variable in place, which reads as a
false equation if written as mathematics (`x = 0.84x + 0.16y` would force `x = y`). The chain
collapses *exactly* into a single flat weighted sum; it is staged only because two intermediate
vectors are consumed later as **guides**.

| # | stage | formula | constants |
|---|---|---|---|
| 1 | parent | `0.525·maek + 0.175·mixpf + 0.300·st_heavy` | derived from `MIXPF`, `PUBLIC_HEAVY` |
| 2 | WARP overlay | `0.80·parent + 0.20·warplookup` | `WARP` |
| 3 | corrections | `0.78·candidate + .05·r_dp + .04·r_vw + .02·r_cau + .05·p2r + .03·r_wlvl + .03·r_wmix` | `CORRECTION_MEMBERS`; parent factor **derived** as `1 − Σ` |
| 4 | newfeats | `0.84·candidate + 0.16·newfeats` | `NEWFEATS_W` |
| 5 | combo B | `0.9408·candidate + 0.0192·dp_rate_ens + 0.0400·r_vw_pp` | `B_DPRE_W`, `B_VWPP_W`; residual **derived** |
| 6 | gain | `anchor + 1.12·(candidate − anchor)` | `GAIN` |
| 7 | capped dcorr | `+ GAIN·(1−B_DPRE_W−B_VWPP_W)·_HEAVY_EFF · clip(dcorr − stock, ±10 ft)` | `DCORR_CAP`; `_HEAVY_EFF` **derived** |
| 8 | U-space smoother | gaussian σ=80 rows @ 0.80, then IRLS degree-6 polynomial @ 0.30 | — |
| 9 | overlay | `apply_leak(candidate)` — exact truth on wells present in `train/` | — |

Every residual weight is **derived** (`1 − Σ others`) rather than written out, so the weights
cannot silently stop summing to 1. A flattened list would give that guarantee up.

**maek and mixpf are DECOUPLED members**, not a nested slot: they enter the parent at derived
weights as independent engines, and the kernel asserts the decoupled form reproduces the legacy
nested arithmetic to 1e−9 ft.

`candidate_preoverlay_submission.csv` is written **before** step 9 and is the only leak-free
artifact the run emits. It is the regression target for every refactor: if it moves, the
refactor changed the model.

## The two guides

Staging exists because two intermediates are named predictions that later members consume:

- `public_parent` (after stage 1) guides `r_dp`, `r_cau`, `r_wlvl` and `r_wmix`.
- `candidate_preoverlay` (after stage 2) guides `r_vw` and `r_vw_pp`, is the fallback every
  decoder reverts to on a failed well, and is the fill value for `p2r`.

## Stage map

| cell | stage | contents |
|---|---|---|
| 1–2 | — | config constants, helpers, and the blend invariants (asserted before any compute) |
| 4 | 0b | module loader: six pipeline `.py` files, each SHA-256 verified |
| 6 | A | mixpf decoder definitions |
| 8, 10 | B, B2 | WARP model + 33-feature builder, then inference over both nets × 5 folds |
| 12 | C | **maek** inference — the cached 186-feature LightGBM |
| 14, 16 | D, E | STRIDE aligner definitions, then the per-well decode (`st_heavy` + `p2r`) |
| 19 | G1 | member assembly onto the shared row order |
| 21 | G2 | blend chain, stages 1–2, plus the decoupling regression guard |
| 23 | G3 | correction weights + the two guided decoders |
| 25 | G4 | per-well decode, then the correction fold-in |
| 27, 29 | G4b | `dp_rate_ens` — rate-space DP definition, then the per-well decode |
| 31 | G5a | newfeats + combo B |
| 33 | G5b | amplitude gain + capped dcorr |
| 35 | G5c | U-space smoother (gaussian + IRLS polynomial) |
| 37 | G5d | overlay, degeneracy guards, submission, artifacts |

These indices are checked by `test_stage_map_matches_the_notebook`: splitting a cell shifts
every index after it, and this table has silently gone stale once already.

Stage letter `F` is retired; the lettering is kept as a historical anchor so old kernel logs
stay comparable.

## Provenance

Every artifact derives from the competition data alone; no external data is used.

| artifact | produced by |
|---|---|
| `maek-ms-model` | `maek-ms-model` — a 7-cell trainer that asserts its OOF against 7.895818. Its models were verified numerically identical to the shipped ones, fold by fold, at the tree level. |
| `chassis-modules` | `chassis_modules_clean/` in the repository |
| `warplookup-module` | `warplookup_defs.py` — pinned separately, ships in its own dataset |
| `chassis-config` | `chassis_ds/` in the repository |
| WARP fold weights | `kernel_warp_direct_aug_f0..f4/` |

`st_heavy`, `p2r` and all six `r_*` corrections are computed inside this run from the modules
and the competition data. They carry no trained weights: given the modules, they reproduce
exactly.

## Two things that look odd and are not

The maek module ends in a short **scaffold** rather than a model. That is deliberate: a second
pipeline ("M6") once lived there, contributed nothing numerically, and was deleted along with
~880 lines and 21 duplicate definitions. What remains is load-bearing — it builds `test_df`,
defines `apply_leak`, sets `COMP`, and writes the file that *is* the maek member of the blend.
Weight zero did not imply removable, because the plumbing was shared.

`maek_infer.py` sits unused in `chassis-modules`. It is not dead either: kernels
v24–v28 load it, and v29 superseded it with `maek_own.py`. It stays so those runs reproduce.

## v31 — review fixes

v31 is v29 with a four-pass review applied. Three real bugs: the dcorr delta was written
before the assert that validates it (a failed well could inject a correction for a member
absent from its blend); the `[GAIN]` log line collapsed to a formula in `anchor/GAIN − 2·anchor`
and printed 153 ft; the `[NEWFEATS]` log measured the post-update vector and overstated its
step 5.25×. One doctrine fix: the dcorr coefficient's `0.1572` was hand-typed and is now
derived and asserted. The rest is dead constants, stale names and structure. **`DCORR_CAP`
stays at 10 ft.** Only the derived coefficient changes any number, by ≤5.1e−4 ft per row.


In [ ]:
# ===== v28: make the run bit-reproducible =====================================
# Measured on v27: re-running IDENTICAL code drifts by 1.305e-05 ft on every row. Diffing
# per member isolates it exactly -- maek, mixpf, stride_heavy and public70_parent are all
# bit-stable; only warp_ensemble (3.9e-04) and warp_newfeats (1.95e-04) move. Both are
# Keras nets doing inference off fixed weights, so this is reduction/thread order, not
# seeding. These flags must be set BEFORE tensorflow is imported anywhere -- the WARP
# modules import it in their own headers, so the first code cell is the only safe place.
import os as _os
_os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
_os.environ.setdefault("TF_CUDNN_DETERMINISTIC", "1")

# ============================ SMOKE TESTS ============================
# Fail the run in seconds if a blend invariant is broken, rather than producing a
# plausible-looking wrong submission 30+ minutes later.
#
# Weights live in ONE place (chassis/config.py) with the parent weight DERIVED, so a
# member cannot be added without the residual adjusting. These asserts guard the
# registry itself. The full suite (103 tests: golden regression, mutation-verified
# invariants, and the v31 review-fix locks) runs locally under `pytest chassis/`.
# It targets whichever chassis chassis/current_chassis.txt names.
import sys, glob

_cfg = [p for p in glob.glob('/kaggle/input/**/config.py', recursive=True)
        if (__import__('pathlib').Path(p).parent / 'selfcheck.py').exists()]
if _cfg:
    sys.path.insert(0, str(__import__('pathlib').Path(_cfg[0]).parent))
    from selfcheck import run_smoke_tests
    import config as CFGW
    run_smoke_tests()
    W_MIXPF_CFG          = CFGW.MIXPF.value
    PUBLIC_MAEK_CFG      = CFGW.PUBLIC_MAEK.value
    PUBLIC_HEAVY_CFG     = CFGW.PUBLIC_HEAVY.value
    WARP_REUSE_CFG       = CFGW.WARP.value
    GAIN_CFG             = CFGW.GAIN
    NEWFEATS_W_CFG       = CFGW.NEWFEATS.value if CFGW.NEWFEATS.enabled else 0.0
    CORRECTION_CFG       = {m.name: m.value for m in CFGW.CORRECTION_MEMBERS}
    PARENT_MAEK_CFG      = CFGW.PARENT_MAEK
    PARENT_MIXPF_CFG     = CFGW.PARENT_MIXPF
    PARENT_HEAVY_CFG     = CFGW.PARENT_HEAVY
else:
    print('[SMOKE] chassis config dataset not attached - falling back to inline constants')
    W_MIXPF_CFG          = 0.25
    PUBLIC_MAEK_CFG      = 0.70
    PUBLIC_HEAVY_CFG     = 0.30
    WARP_REUSE_CFG       = 0.20
    NEWFEATS_W_CFG       = 0.16
    GAIN_CFG             = 1.12
    CORRECTION_CFG = {'dp':0.05,'vw':0.04,'cau':0.02,'p2r':0.05,'wlvl':0.03,'wmix':0.03}
    assert abs(sum(CORRECTION_CFG.values()) - 0.22) < 1e-12
    assert abs(PUBLIC_MAEK_CFG + PUBLIC_HEAVY_CFG - 1.0) < 1e-12
    # DERIVED products, never hand-typed: a 1e-4 weight error is 1.29 ft of bias per row.
    PARENT_MAEK_CFG      = PUBLIC_MAEK_CFG * (1.0 - W_MIXPF_CFG)
    PARENT_MIXPF_CFG     = PUBLIC_MAEK_CFG * W_MIXPF_CFG
    PARENT_HEAVY_CFG     = PUBLIC_HEAVY_CFG
    assert abs(PARENT_MAEK_CFG + PARENT_MIXPF_CFG + PARENT_HEAVY_CFG - 1.0) < 1e-12
    print('[SMOKE] inline invariants held')
# =====================================================================

In [ ]:
# ============================ LOGGING ============================
# One scheme for the whole run. The kernel log is the only forensic record of a hidden
# rerun, so every stage announces itself, carries elapsed time, and summarises the vectors
# it produced. A member that arrives half-NaN is then visible in the log, not three stages
# later as an assertion failure.
import time as _time
import numpy as _np

_T_START = _time.time()
_W = 78


def _elapsed():
    s = int(_time.time() - _T_START)
    return f"{s // 60:02d}:{s % 60:02d}"


def stage(name, detail=""):
    """Announce a pipeline stage with a banner and cumulative elapsed time."""
    head = f" STAGE {name} " + (f"- {detail} " if detail else "")
    print("\n" + "=" * _W, flush=True)
    print(f"{head}{'.' * max(2, _W - len(head) - 9)} [+{_elapsed()}]", flush=True)
    print("=" * _W, flush=True)


def info(msg):
    print(f"   {msg}", flush=True)


def stat(label, value):
    print(f"   {str(label):<30} {value}", flush=True)


def ok(msg):
    print(f"   [OK] {msg}", flush=True)


def warn(msg):
    print(f"   [!!] {msg}", flush=True)


def eval_rows(hw):
    """Positional indices of the eval corridor, and the last known row.

    ONE definition. The four call sites used to carry four copies of this arithmetic, and a
    2026-08-04 check confirmed they agreed on all 773 wells -- by CONVENTION, not by
    construction: one copy used the pandas `.index.max()` of the notna frame while the
    others used positional `flatnonzero`, which coincide only because `pd.read_csv` yields a
    RangeIndex. Any upstream reindex or row filter would have desynchronised them silently,
    and the members would have been assembled against different row sets.

    The `ev[0] != last + 1` branch handles a well with known rows AFTER the first unknown.
    That fires on 0 of 773 training wells; it is kept because the competition files are not
    contractually one contiguous corridor.
    """
    kn = hw["TVT_input"].notna().to_numpy()
    ev = _np.flatnonzero(~kn)
    last = int(_np.flatnonzero(kn)[-1])
    if len(ev) and ev[0] != last + 1:
        ev = ev[ev > last]
    return ev, last


def vec(label, a, unit="ft"):
    """One-line summary of a member vector: coverage first, because that is what breaks."""
    a = _np.asarray(a, dtype=float)
    fin = _np.isfinite(a)
    if not fin.any():
        warn(f"{label:<18} ALL NON-FINITE ({a.size:,} rows)")
        return
    print(f"   {label:<18} n={a.size:>7,}  finite={100 * fin.mean():6.2f}%  "
          f"mean={_np.nanmean(a):>10.2f}  sd={_np.nanstd(a):>8.2f} {unit}", flush=True)


stage("0", "environment")
info("chassis: modules load SHA-pinned from a dataset; weights from the tested registry")


## Stage 0 — load the pipeline modules

Six real `.py` files, SHA-pinned. Previously ~248k chars of escaped string literals.
Five ship in `chassis-modules`; `warplookup_defs.py` ships in
`warplookup-module`, which is why its hash is pinned separately.


In [ ]:
stage("0b", "load + SHA-verify the pipeline modules")
# ============================ MODULE LOADER ============================
# The six pipeline modules used to be embedded as escaped string literals -- ~248k
# chars of Python with no highlighting, no linting, no diffable history, and edits that
# required unicode_escape round-trips (which is how a mismatched model class nearly
# shipped). They now live as real .py files in the `chassis-modules` dataset.
#
# Execution semantics are UNCHANGED: each source is still exec'd exactly as before, into
# the same namespace as before. Only the storage moved. That is what makes this
# refactor verifiable against the golden fixture.
#
# Each file is SHA-pinned: if the attached dataset version does not match what this
# notebook was validated against, the run dies here instead of silently computing
# something else.
import hashlib as _hl, glob as _gl
from pathlib import Path as _P

_MOD_SHA = {
    'mixpf_pipe.py': 'c561547bd27ec88d701fe9260800c1c794dc169f29a1bfd6cb9c728479548abf',
    'warp_costvolume.py': 'b95e0885d4c3e8877e9cdffd41836483b8c24ab71a9213218149dd3c3ff7ac0f',
    'maek_own.py': 'd40bc0c069510b51d66ff9620e9955f8805b89b30e664c258c49fa66a70154f7',
    'stride_defs.py': 'b3cc815105f7dd72ba72602af54169665135e0b4b477808b9d78d5e80d454a41',
    'warp_direct.py': '335b4ab00f7e6001104c92144f4c2911c5afdffdf480c87b7049d2915c4c614d',
}

def _load_module_source(filename):
    hits = _gl.glob(f'/kaggle/input/**/{filename}', recursive=True)
    assert hits, (f'{filename} not found -- attach chassis-modules '
                  '(or warplookup-module for warplookup_defs.py)')
    text = _P(hits[0]).read_text()
    got = _hl.sha256(text.encode()).hexdigest()
    want = _MOD_SHA[filename]
    assert got == want, (
        f'{filename} SHA mismatch\n  expected {want}\n  found    {got}\n'
        'The attached dataset version differs from the validated one. Refusing to run.')
    print(f'[MODULES] {filename:18} {len(text):>7,} chars  sha {got[:10]} OK', flush=True)
    return text

# warplookup_defs.py is pinned separately because it ships in a DIFFERENT dataset
# (warplookup-module), not in chassis-modules with the other five.
_MOD_SHA['warplookup_defs.py'] = '333b1ee306cf97863028763b080f40cb0bcd144604ee33eb9c2396a2671f315e'
_MIXPF_SOURCE = _load_module_source('mixpf_pipe.py')
WARP_SOURCE = _load_module_source('warp_costvolume.py')
WARP_ORIG_SOURCE = _load_module_source('warp_direct.py')
MAEK_SOURCE = _load_module_source('maek_own.py')

# maek artifact routing is BUILT INTO maek_own.py (hint-sorted glob + assert), so the
# v19 source-patch that injected it is gone -- its anchor no longer exists.
DCORR_PATCH = (
    '    _wd = float(os.environ.get("W_DCORR", "0"))\n    if _wd > 0 and len(score_idx) >= 5:\n        _do = np.diff(obs)\n        _ds = np.diff(sim, axis=1)\n        _do = _do - _do.mean()\n        _ds = _ds - _ds.mean(axis=1, keepdims=True)\n        _den = (np.sqrt((_do * _do).sum()) * np.sqrt((_ds * _ds).sum(axis=1))) + 1e-9\n        gr_loss = gr_loss + _wd * (1.0 - (_ds @ _do) / _den)\n'
)
CAP_PATCH = (
    '\n\n# ===== v22b: the dcorr path, decoded SEPARATELY (decode_one_well stays pristine) =====\n# Wrapping decode_one_well would propagate through public70_parent into the r_vw_pp\n# corrector (measured: up to 15 ft), which the bank gate held fixed. Everything the\n# chassis calls therefore keeps the stock path; only this helper enables the term.\ndef _decode_dcorr(hw, tw, *args, **kwargs):\n    os.environ["W_DCORR"] = "0.60"\n    try:\n        return decode_one_well(hw, tw, *args, **kwargs)\n    finally:\n        os.environ["W_DCORR"] = "0"\n'
)
STRIDE_SOURCE = _load_module_source('stride_defs.py')

# ===== STRIDE emission gains the delta-pattern CORRELATION term (OFF by default) =====
# Correlating the DERIVATIVES asks "do the stripes wiggle where the typewell says", which
# pointwise level matching cannot answer. Standalone 9.9472 -> 9.0965. Enabled only via
# _decode_dcorr; W_DCORR defaults to 0 so the shipped members are unchanged.
_v22_gr = "    gr_loss = np.mean(np.log1p(d * d), axis=1)\n"
assert STRIDE_SOURCE.count(_v22_gr) == 1, 'stride emission anchor moved -- re-derive'
STRIDE_SOURCE = STRIDE_SOURCE.replace(_v22_gr, _v22_gr + DCORR_PATCH)
STRIDE_SOURCE = STRIDE_SOURCE + CAP_PATCH
WARPLOOKUP_SOURCE = _load_module_source('warplookup_defs.py')
print('[MODULES] all 6 pipeline modules loaded and SHA-verified', flush=True)
# =======================================================================

## Stage A — mixpf decoder

In [ ]:
stage("A", "mixpf decoder")
# _MIXPF_SOURCE is loaded by the MODULE LOADER cell (SHA-pinned)
# Clean WIGGLE/TREND replica. No train-contact overwrite is applied.
import hashlib, os, time            # `json` was imported here and never used
from pathlib import Path
import numpy as np, pandas as pd
T0 = time.time()
COMP = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
test_files = sorted((COMP/'test').glob('*__horizontal_well.csv'))
# REVIEW FIX (v31): WARP_WEIGHT / WARP_PLACEMENT were dead -- read only by the print on
# the next line, which also misdescribed the pipeline (it announced a "slot swap to
# newfeats" when the slot is warplookup at WARPLOOKUP_W=1.0 and the overlay weight comes
# from the config registry). Removed; the real weights are logged where they are applied.


## Stage B — WARP: model + 33-feature builder

In [ ]:
stage("B", "WARP model + feature builders")
# WARP_SOURCE / WARP_ORIG_SOURCE are loaded by the MODULE LOADER cell (SHA-pinned).
#
# CHANGED IN v5 -- the two nets are NO LONGER exec'd into globals().
# Both modules define `make_sample`, `H_FEATURES` and `model` at top level, so exec'ing
# them into one namespace makes the second silently clobber the first. Each now gets its
# own dict namespace (see the WARP INFERENCE cell). The only coupling to the outer scope
# is `coord_mean`/`coord_std`, which make_sample reads from ITS OWN module globals
# (warp_costvolume.py:237) -- so they are injected into each namespace after exec, which is
# exactly what the old globals() exec was accomplishing implicitly.
# The notebook used to inherit its ENVIRONMENT from this exec as a side effect: both
# WARP modules open with the same header (numpy/pandas/tf + COMP/TRAIN/OUT), so
# exec'ing either into globals() silently populated the downstream cells. Removing the
# globals() exec removed that too -- which is what broke v5 run 1 with
# "NameError: name 'TRAIN' is not defined". Static analysis of the whole notebook says
# exactly two names came from there and nowhere else: TRAIN and tf. Declared explicitly
# now, so the dependency is visible instead of accidental.
import tensorflow as tf

# v28 determinism: pin op determinism and single-threaded op scheduling for the WARP nets.
# enable_op_determinism() must run before any op executes. Only TF is constrained -- the
# LightGBM/numpy members are already bit-stable, so their threading is left alone.
#
# MEASURED 2026-08-04, AND IT DOES NOT WORK. Two runs of numerically identical code with
# this block active and printing success still differ: warp_ensemble 3.906e-04 ft,
# warp_newfeats 1.953e-04 ft -> candidate_preoverlay 1.271e-05 ft. Those are the same
# members and the same magnitudes v27 measured BEFORE this block existed. It is retained
# because it is harmless and may constrain something we have not isolated, but do NOT
# treat its success message as a reproducibility guarantee -- see the master doc.
try:
    tf.config.experimental.enable_op_determinism()
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.utils.set_random_seed(42)
    print("[DETERMINISM] TF op-determinism on, intra/inter op threads pinned to 1", flush=True)
except Exception as _e:
    print(f"[DETERMINISM] could not fully enable: {type(_e).__name__}: {_e}", flush=True)

from pathlib import Path as _Path
# Static analysis of the whole notebook says exactly TWO names came from the old
# globals() exec and nowhere else: TRAIN and tf. COMP is re-declared here for locality.
# `OUT` was declared alongside them and never read -- the modules define their own OUT in
# their own exec namespaces -- so it is gone (5-pass review, dead store).
COMP  = _Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
TRAIN = COMP / 'train'
assert TRAIN.exists(), TRAIN

open('/kaggle/working/_warp_costvolume.py','w').write(WARP_SOURCE)
open('/kaggle/working/_warp_direct.py','w').write(WARP_ORIG_SOURCE)
print('[WARP] sources staged; nets are built per-namespace in the inference cell', flush=True)

## Stage B2 — WARP inference (5 folds)

In [ ]:
stage("B2", "WARP inference, both nets x 5 folds")

# Coordinate normalization is the exact full-train, target-free transform used by every fold.
coord_sum = np.zeros(3, float); coord_sum2 = np.zeros(3, float); coord_n = 0
for path in sorted(TRAIN.glob('*__horizontal_well.csv')):
    coords = pd.read_csv(path, usecols=['X','Y','Z']).to_numpy(float)
    coord_sum += np.nansum(coords, axis=0); coord_sum2 += np.nansum(coords**2, axis=0)
    coord_n += len(coords)
# Round A of the second audit: every other division in the chassis is guarded (grsc
# and friends by +1e-6, _dm by `_m = _dm > 0`, _span by an explicit >1.0 check, dmd by
# max(...,1.0)). This one was not. It needs zero train files to divide by zero, which the
# TRAIN.exists() assert above makes unreachable -- but "unreachable" is an argument, and a
# one-line assert is a fact.
assert coord_n > 0, 'no train rows found for WARP coordinate normalisation'
coord_mean = coord_sum / coord_n
coord_std = np.sqrt(np.maximum(coord_sum2 / coord_n - coord_mean**2, 1.0))


# ─────────────────────────────────────────────────────────────────────────────────────
# v5: TWO WARP nets, each in its own exec namespace.
#
#   production  WarpDirect  + original 33 features   -> the incumbent slot content
#   newfeats    WarpCanon   + 9 maek physics features -> added as an independent member
#
# Nested GroupKFold(5), 772 wells, ref 7.1000: swapping the slot to newfeats scores
# -0.053, but ADDING newfeats beside the production net scores -0.127. rho between the
# two nets is only 0.792, so the incumbent still carries signal the new net does not.
# Control: adding a SECOND COPY of the production net is worth just -0.010, which is what
# rules out "the blend simply wanted more NN weight". See chassis/config.py STAGE 3.
# ─────────────────────────────────────────────────────────────────────────────────────
def run_warp_net(source, tag, glob_pattern, expect_params=None):
    """Build one WARP net in an isolated namespace and average its 5 folds over the test wells."""
    ns = {}
    exec(compile(source, f'/kaggle/working/_{tag}.py', 'exec'), ns)
    ns['coord_mean'] = coord_mean          # make_sample reads these from ITS module globals
    ns['coord_std']  = coord_std
    make_sample_fn, net = ns['make_sample'], ns['model']
    if expect_params is not None:
        assert net.count_params() == expect_params, (tag, net.count_params(), expect_params)

    samples = []
    for path in test_files:
        well = path.name.replace('__horizontal_well.csv','')
        hw = pd.read_csv(path).reset_index(drop=True)
        tw = pd.read_csv(path.parent/f'{well}__typewell.csv')
        ev = np.flatnonzero(hw['TVT_input'].isna().to_numpy())
        boundary = int(ev[0])-1
        fake = hw.copy()
        anchor = float(fake['TVT_input'].iat[boundary])
        fake['TVT'] = fake['TVT_input'].fillna(anchor)
        sample = make_sample_fn({'well':well,'hw':fake,'tw':tw}, boundary,
                                real_boundary=True, allow_short=True)
        assert sample is not None, f'{tag}: WARP sample coverage failure: {well}'
        samples.append(sample)

    paths = sorted(glob.glob(glob_pattern, recursive=True))
    assert len(paths) == 5, (tag, paths)
    hashes = {Path(p).name: hashlib.sha256(open(p,'rb').read()).hexdigest() for p in paths}
    acc = {}
    for k, wp in enumerate(paths, 1):
        net.load_weights(wp)
        for sample in samples:
            inputs = (
                tf.convert_to_tensor(sample['h'][None], tf.float32),
                tf.convert_to_tensor(sample['tw'][None], tf.float32),
                tf.convert_to_tensor(sample['geo'][None], tf.int32),
                tf.convert_to_tensor(sample['tw_mask'][None], tf.bool),
                tf.convert_to_tensor(sample['dmd'][None], tf.float32),
                tf.convert_to_tensor([sample['anchor_tvt']], tf.float32),
            )
            # warpx50-lineage nets (warplookup) take a 7th FiLM conditioning input.
            # The two incumbent nets do not -- detect from the sample, never assume.
            if 'cond' in sample:
                inputs = inputs + (tf.convert_to_tensor(sample['cond'][None], tf.float32),)
            pred = net(inputs, training=False)[0][0].numpy().astype(float)
            acc[sample['well']] = acc.get(sample['well'], np.zeros(len(pred))) + pred
        print(f'[WARP:{tag}] model {k}/5  {Path(wp).name}', flush=True)
    out = {w: v/len(paths) for w, v in acc.items()}
    del net, samples
    tf.keras.backend.clear_session()
    import gc; gc.collect()
    return out, hashes

# ---- Idea A net: dynamic cost volume + orientation + learned gate ----------------
# pooled 5-fold OOF 9.5865 -- best standalone NN in the bank (warpx50 9.9588, newfeats
# 10.0120, production warp 10.3490). Enters the WARP slot; WARPLOOKUP_W sets how much.
# holdout-verified: w=0.5 -> 6.9157 (P(help) 96%), w=1.0 -> 6.8888 (P(help) 90%),
# baseline 6.9729. The global scale stays 1.12 (optimum moves only 1.1198 -> 1.1130).
WARPLOOKUP_W = float(__import__('os').environ.get('WARPLOOKUP_W', '1.0'))
assert 0.0 <= WARPLOOKUP_W <= 1.0, WARPLOOKUP_W
lookup_predictions, lookup_hashes = run_warp_net(
    WARPLOOKUP_SOURCE, 'warplookup', '/kaggle/input/**/warplookup2_f[0-4].weights.h5')

warp_predictions,  weight_hashes  = run_warp_net(
    WARP_ORIG_SOURCE, 'warp_prod', '/kaggle/input/**/warp_direct_aug_f[0-4].weights.h5')
newfeats_predictions, newfeats_hashes = run_warp_net(
    WARP_SOURCE, 'warp_newfeats', '/kaggle/input/**/warpcanon_f[0-4]_newfeats.weights.h5')

_pw = set(warp_predictions); _nw = set(newfeats_predictions)
assert _pw == _nw, ('WARP well-set mismatch', _pw ^ _nw)
for _w in _pw:
    assert len(warp_predictions[_w]) == len(newfeats_predictions[_w]), ('length mismatch', _w)
_md = float(np.mean([np.mean(np.abs(warp_predictions[w]-newfeats_predictions[w])) for w in _pw]))
print(f'[WARP] both nets done; mean|prod - newfeats| = {_md:.3f} ft over {len(_pw)} wells', flush=True)
print('[CHASSIS] WARP inference complete before incumbent pipeline', round(time.time()-T0), flush=True)


## Stage C — maek inference (the cached 186-feature LightGBM)

In [ ]:
stage("C", "maek inference - cached 186-feature LightGBM")
# MAEK_SOURCE is loaded by the MODULE LOADER cell (SHA-pinned)
maek_path='/kaggle/working/_maek_clean.py'
open(maek_path,'w').write(MAEK_SOURCE)
g_maek={'__name__':'__main__','__file__':maek_path}
exec(compile(MAEK_SOURCE,maek_path,'exec'),g_maek)
import shutil
shutil.copy('/kaggle/working/submission.csv','/kaggle/working/maek_clean.csv')
print('[CHASSIS] clean MAEK complete', round(time.time()-T0), flush=True)


## Stage D — STRIDE heavy/lite

In [ ]:
stage("D", "STRIDE decoder definitions")
# STRIDE_SOURCE is loaded by the MODULE LOADER cell (SHA-pinned)
# REVIEW FIX (v31): os.environ['MAX_WELLS']='3' used to sit here with no comment.
# stride_defs.py has no __main__ guard and never decodes at import -- MAX_WELLS only
# truncates a TRAIN_FILES list this chassis never reads (it decodes test wells explicitly
# in stage E). Its sole effect was to make one import-time log line read "train wells=3".
# Removed; if a future change makes the module's TRAIN_FILES load-bearing, that must be
# explicit rather than inherited from a stray env var.
stride_path='/kaggle/working/_stride_defs.py'
open(stride_path,'w').write(STRIDE_SOURCE)
g_heavy={'__name__':'__main__','__file__':stride_path}
exec(compile(STRIDE_SOURCE,stride_path,'exec'),g_heavy)
print('[CHASSIS] STRIDE definitions loaded', flush=True)


## Stage E — STRIDE decode per well

Runs the stock decoder for `st_heavy`, the `W_DCORR=0.60` decoder for the capped dcorr
delta, and a trend-override re-decode for `p2r`. A well that fails any of the three is
recorded in `decode_failures` and contributes nothing rather than a partial member.


In [ ]:
stage("E", "STRIDE decode per well (heavy + p2)")

assert g_maek['COMP'] == COMP
heavy, decode_failures = {}, []
dcorr_failures = []          # dcorr decode failures, kept SEPARATE from decode_failures:
                             # a dcorr failure costs only the correction, not the member
p2_failures = []             # likewise for the p2r re-decode
dcw = {}
# v36d CAPPED GAMBLE: dcorr re-enabled at the CHEAPEST measured dose. Board cost of cap 6 is
# +0.002 (v34b 5.932 vs v21 5.930); nested CV gain on top of v36a is -0.0769 (-0.0997 -> -0.1766).
# Everything else identical to v36a, which scored 5.897. Capped dcorr RAISES the win rate
# (58.5% vs 58.0%), keeps ZERO wells damaged >2 ft, and gives b151 99.8% -- the best measured.
# NOT the uncapped variant, whose gain is decorrelated tie-breaking (49.4% win, -0.007 ft median).
DCORR_CAP = float(os.environ.get('DCORR_CAP', '6.0'))
p2w = {}
for number, path in enumerate(test_files, 1):
    well = path.name.replace('__horizontal_well.csv','')
    try:
        _, hw, tw = g_heavy['load_pair'](path)
        ev, last = eval_rows(hw)
        pred_heavy, _, _, _ = g_heavy['decode_one_well'](hw, tw)
        # v22b: the capped dcorr displacement. decode_one_well above is the STOCK path,
        # so heavy/p2r/parent/r_vw_pp are untouched; only this delta is new.
        # REVIEW FIX (v31): this assert used to sit AFTER the dcw write, so a well where it
        # tripped kept a dcorr correction for a member that was not in its blend.
        assert len(ev) == len(pred_heavy)
        # AUDIT FIX (v31, round C): st_heavy is a SHIPPED member carrying 0.30 of the
        # parent; the dcorr delta is an optional correction bounded at +/-DCORR_CAP. The
        # dcorr decode used to run FIRST, so if it raised, the whole well went to
        # decode_failures and `heavy[well]` was never written -- discarding a member that
        # had already decoded successfully and dropping that well to maek/mixpf. An
        # optional correction must never be able to destroy the primary member. Write
        # heavy first, then attempt dcorr under its own handler.
        heavy[well] = pd.Series(pred_heavy, index=ev)
        try:
            _pdc, _, _, _ = g_heavy['_decode_dcorr'](hw, tw)
            _n_dc = min(len(_pdc), len(pred_heavy))
            dcw[well] = pd.Series(np.clip(np.asarray(_pdc, float)[:_n_dc]
                                          - np.asarray(pred_heavy, float)[:_n_dc],
                                          -DCORR_CAP, DCORR_CAP), index=ev[:_n_dc])
        except Exception as _dce:
            # No dcw entry => dcorr_delta stays 0 for this well => the blend simply keeps
            # its stock st_heavy. The >=90% budget in G1 still catches systematic failure.
            dcorr_failures.append((well, type(_dce).__name__, str(_dce)))
            print('[DCORR] decode failed, keeping stock st_heavy:',
                  dcorr_failures[-1], flush=True)
        z_ev = hw['Z'].to_numpy(float)[ev]
        md_ev = hw['MD'].to_numpy(float)[ev]
        S1 = np.asarray(pred_heavy, float) + z_ev[:len(pred_heavy)]
        tp1 = float(np.clip(np.median(np.diff(S1) / np.maximum(np.diff(md_ev[:len(S1)]), 1e-6)), -0.08, 0.08))
        # Same principle as the dcorr handler above: p2r is a 0.05-weight correction with
        # its own fallback (`p2_filled` reverts to the base). It used to share the outer
        # try, so a p2 failure marked the WELL as a STRIDE failure even though st_heavy had
        # decoded fine -- inflating _fail_frac toward the 20% budget for a reason that has
        # nothing to do with st_heavy. Now `decode_failures` means exactly one thing:
        # st_heavy could not be produced.
        try:
            a_tr = g_heavy['recent_s_trend'](hw, int(last))
            pred_p2, _, _, _ = g_heavy['decode_one_well'](hw, tw, trend_override=0.5 * (a_tr + tp1))
            p2w[well] = pd.Series(pred_p2, index=ev)
        except Exception as _p2e:
            p2_failures.append((well, type(_p2e).__name__, str(_p2e)))
            print('[P2] re-decode failed, p2r falls back to the base:',
                  p2_failures[-1], flush=True)
    except Exception as error:
        decode_failures.append((well, type(error).__name__, str(error)))
        print('[CHASSIS] STRIDE failure', decode_failures[-1], flush=True)
    if number % 20 == 0 or number == len(test_files):
        print('[CHASSIS] STRIDE', number, '/', len(test_files), 'elapsed', round(time.time()-T0), flush=True)

# ---- failure budget: STRIDE ---------------------------------------------------------
# A failed well leaves heavy_values NaN and the parent silently falls back to maek-only for
# that well. On 3 public wells this never fires, so the bitwise gate cannot see it; on 151
# hidden wells it would degrade the blend invisibly. Warn on ANY failure, fail only on a
# systematic one -- a hard assert here would turn a partial degradation into a zero score.
_n_wells = len(test_files)
_fail_frac = len(decode_failures) / max(_n_wells, 1)
if decode_failures:
    warn(f"STRIDE fell back on {len(decode_failures)}/{_n_wells} wells "
         f"({100 * _fail_frac:.1f}%): {[w for w, _, _ in decode_failures][:8]}")
else:
    ok(f"STRIDE decoded all {_n_wells} wells")
assert _fail_frac <= 0.20, (
    f"FATAL: STRIDE failed on {len(decode_failures)}/{_n_wells} wells "
    f"({100 * _fail_frac:.1f}%) -- the blend would be maek-only for most of the field")



## Stage G — assembly, gain, overlay, submission

## Stage G1 - member assembly and submission helpers

Gathers every member vector onto the shared row order and defines the two helpers used later.
`ordered_submission` is the only writer of a submission frame: it re-joins onto the competition
sample so a row-order change anywhere upstream becomes an assertion failure rather than a silently
misaligned file.


In [ ]:
stage("G1", "member assembly")

from pathlib import Path as _Path
import hashlib as _hashlib
import json as _json
import numpy as np
import pandas as pd

test_df = g_maek['test_df']
assert np.array_equal(test_df.index.to_numpy(), np.arange(len(test_df))), 'test_df index is not positional'
ids = test_df['id'].astype(str).to_numpy()
assert len(ids) == len(set(ids)), 'test ids are not unique'
wellcol = test_df['id'].str.rsplit('_', n=1).str[0].to_numpy()
ridx = test_df['id'].str.rsplit('_', n=1).str[1].astype(int).to_numpy()
assert np.array_equal(wellcol, test_df['well'].astype(str).to_numpy()), 'id/well mismatch'

maek = pd.read_csv('/kaggle/working/maek_clean.csv')
assert maek['id'].is_unique and maek['tvt'].notna().all(), 'MAEK artifact integrity'
maek_map = dict(zip(maek['id'].astype(str), maek['tvt']))
assert all(value in maek_map for value in ids), 'MAEK id coverage mismatch'
maek_values = np.array([maek_map[value] for value in ids], dtype=np.float64)

heavy_values = np.full(len(ids), np.nan, dtype=np.float64)
warp_values = np.full(len(ids), np.nan, dtype=np.float64)
newfeats_values = np.full(len(ids), np.nan, dtype=np.float64)
for well, values in heavy.items():
    select = wellcol == well
    heavy_values[select] = values.reindex(ridx[select]).to_numpy(dtype=np.float64)
p2_values = np.full(len(ids), np.nan, dtype=np.float64)
dcorr_delta = np.zeros(len(ids), dtype=np.float64)   # v22b: 0 = no change
_dc_wells = 0
for well, values in dcw.items():
    select = wellcol == well
    _v = values.reindex(ridx[select]).to_numpy(dtype=np.float64)
    dcorr_delta[select] = np.nan_to_num(_v, nan=0.0)
    _dc_wells += 1
print('[DCORR] capped delta on %d/%d wells; max |move| %.3f ft (cap %.1f), mean %.3f'
      % (_dc_wells, len(set(wellcol)), float(np.abs(dcorr_delta).max()), DCORR_CAP,
         float(np.abs(dcorr_delta).mean())), flush=True)
assert _dc_wells >= 0.90 * len(set(wellcol)), 'dcorr decoded too few wells'
assert float(np.abs(dcorr_delta).max()) <= DCORR_CAP + 1e-9, 'cap not binding'
for well, values in p2w.items():
    select = wellcol == well
    p2_values[select] = values.reindex(ridx[select]).to_numpy(dtype=np.float64)
for well, values in warp_predictions.items():
    _lk = lookup_predictions.get(well)
    _base = np.asarray(values, dtype=np.float64)
    warp_values[wellcol == well] = (
        (1.0 - WARPLOOKUP_W) * _base + WARPLOOKUP_W * np.asarray(_lk, dtype=np.float64)
        if _lk is not None else _base)
for well, values in newfeats_predictions.items():
    newfeats_values[wellcol == well] = np.asarray(values, dtype=np.float64)

assert np.isfinite(maek_values).all(), 'MAEK contains nonfinite values'
assert np.isfinite(warp_values).all(), 'WARP coverage/finite failure'
assert np.isfinite(newfeats_values).all(), 'WARP-newfeats coverage/finite failure'


sample = pd.read_csv(COMP/'sample_submission.csv')[['id']]
sample['id'] = sample['id'].astype(str)
assert sample['id'].is_unique and len(sample) == len(ids), 'sample id uniqueness/length mismatch'
assert set(sample['id']) == set(ids), 'sample/test id set mismatch'

def ordered_submission(values):
    frame = pd.DataFrame({'id': ids, 'tvt': np.asarray(values, dtype=np.float64)})
    assert frame['id'].is_unique and frame['tvt'].notna().all(), 'prediction frame integrity'
    output = sample.merge(frame, on='id', how='left', validate='one_to_one')
    assert len(output) == len(sample) and output['tvt'].notna().all(), 'submission coverage/NaN'
    assert output['id'].equals(sample['id']), 'sample order changed'
    return output

vec("maek", maek_values)
vec("stride_heavy", heavy_values)
vec("warp", warp_values)
vec("warp_newfeats", newfeats_values)
vec("p2r", p2_values)


## Stage G2 - blend chain, stages 1 to 2

Stage 1 builds the parent from three DECOUPLED members, stage 2 overlays WARP:

    1. parent  = 0.525*maek + 0.175*mixpf + 0.300*st_heavy   (falls back to the
                 maek/mixpf slot on wells where STRIDE heavy is NaN)
    2. v1warp  = 0.80*parent + 0.20*warplookup

Between them sits a REGRESSION GUARD, not a pipeline stage: the nested slot form this
decoupling replaced is recomputed and asserted equal per row. The previous version of this
header listed that guard as "stage 2", which disagreed with the stage table in cell 0.

Weights come from the tested registry, one per line -- never tuple-packed, because a regex
for the third name in a packed tuple silently returns the first value.


In [ ]:
stage("G2", "blend chain, stages 1-2")   # the third item under this header was the
                                             # decoupling REGRESSION GUARD, not a stage
# ---- weights come from chassis/config.py: ONE source of truth, tested invariants ----
# The previous form was tuple-packed:
#     PUBLIC_MAEK, PUBLIC_HEAVY, WARP_REUSE_WEIGHT = 0.70, 0.30, 0.20
# which is unreadable and un-greppable: a regex for WARP_REUSE_WEIGHT returns 0.70,
# the FIRST tuple element. One constant per line, sourced from the tested registry.
PUBLIC_MAEK       = PUBLIC_MAEK_CFG      # 0.70  maek slot share of the parent
PUBLIC_HEAVY      = PUBLIC_HEAVY_CFG     # 0.30  STRIDE heavy share of the parent
WARP_REUSE_WEIGHT = WARP_REUSE_CFG       # 0.20  WARP overlay weight
assert abs(PUBLIC_MAEK + PUBLIC_HEAVY - 1.0) < 1e-12, 'parent weights must sum to 1'
# ================= parent: maek, mixpf and st_heavy as DECOUPLED members =================
# OOF (773 wells, maek reproduced from raw): maek_MS 7.9263 -> +0.25*mixpf 7.7676 (nested 7.7861),
# -0.263 vs the 8.0489 baseline. mixpf is ABSENT from the production formula and is worse
# standalone (9.39 vs 8.05) - it earns its weight by decorrelation, not strength.
W_MIXPF = W_MIXPF_CFG                   # 0.25  mixpf share within the maek/mixpf slot
PARENT_MAEK  = PARENT_MAEK_CFG          # 0.525 maek  share of the parent  (derived)
PARENT_MIXPF = PARENT_MIXPF_CFG         # 0.175 mixpf share of the parent  (derived)
PARENT_HEAVY = PARENT_HEAVY_CFG         # 0.300 heavy share of the parent
assert abs(PARENT_MAEK + PARENT_MIXPF + PARENT_HEAVY - 1.0) < 1e-12, 'parent weights must sum to 1'
_mx = Path('/kaggle/working/_mixpf_pipe.py')
_mx.write_text(_MIXPF_SOURCE)
_g_mix = {'__name__': '__main__', '__file__': str(_mx)}
_t_mix = time.time()
exec(compile(_MIXPF_SOURCE, str(_mx), 'exec'), _g_mix)
print('[MIXPF] pipeline done in %.1f min' % ((time.time()-_t_mix)/60.0), flush=True)
_mxdf = pd.read_csv('/kaggle/working/mixpf_rows.csv')
_mix_row = pd.Series(_mxdf['tvt'].to_numpy(), index=_mxdf['id'].to_numpy()).reindex(ids).to_numpy(np.float64)
_cov = float(np.isfinite(_mix_row).mean())
print('[MIXPF] row coverage %.4f' % _cov, flush=True)
assert _cov > 0.999, 'mixpf coverage failure %.4f' % _cov
mixpf_values = _mix_row
# maek_values stays PURE maek from here on -- it used to be OVERWRITTEN by the slot mix,
# which is why components.parquet's 'maek_clean' was the slot rather than maek itself.
# NAMING (v31): this was `_maek_slot_legacy`, which read as dead code. It is LIVE -- it
# is the parent's fallback wherever STRIDE heavy is NaN, and it is the `maek_clean` column
# five research_directions/ scripts read out of components.parquet. Renamed for what it is.
_maek_mixpf_slot = (1.0 - W_MIXPF) * maek_values + W_MIXPF * mixpf_values
print('[MIXPF] maek and mixpf are DECOUPLED members: parent = %.6f*maek + %.6f*mixpf + %.6f*heavy'
      % (PARENT_MAEK, PARENT_MIXPF, PARENT_HEAVY), flush=True)
# =============================================================================================
# DECOUPLED form. maek and mixpf enter as independent engines at derived weights.
# The heavy-NaN fallback keeps the maek:mixpf ratio and renormalises, exactly as before.
public_parent = np.where(
    np.isfinite(heavy_values),
    PARENT_MAEK * maek_values + PARENT_MIXPF * mixpf_values + PARENT_HEAVY * heavy_values,
    _maek_mixpf_slot,
)
# REGRESSION GUARD: the nested form this replaced, recomputed and compared per row.
# A refactor that silently changes the prediction is the failure mode this exists to catch.
_legacy_parent = np.where(
    np.isfinite(heavy_values),
    PUBLIC_MAEK * _maek_mixpf_slot + PUBLIC_HEAVY * heavy_values,
    _maek_mixpf_slot,
)
_dmax = float(np.nanmax(np.abs(public_parent - _legacy_parent)))
assert _dmax < 1e-9, f'DECOUPLING REGRESSION: parent moved by {_dmax:.3e} ft (must be 0)'
print('[DECOUPLE] decoupled parent == nested slot form, max|delta| %.2e ft' % _dmax, flush=True)
candidate_preoverlay = (1.0 - WARP_REUSE_WEIGHT) * public_parent + WARP_REUSE_WEIGHT * warp_values

vec("parent", public_parent)
vec("v1warp (stage 3)", candidate_preoverlay)


## Stage G3 - correction weights and the guided decoders

The six correction members and the two decoders that produce five of them. Both decoders are
DETERMINISTIC: no training target, no fitted parameters. They re-decode the GR barcode against a
guide that is itself a blend, so all of their out-of-fold-ness comes from the guide, not from them.
`_dp_decode` is a Viterbi on raw GR; `_wl_decode` returns the windowed level and mixture members.


In [ ]:
stage("G3", "correction weights + guided decoders")
# ---- guided-DP member (truth-free Viterbi on raw GR, guided toward the maek/heavy parent) ----
# Correction members, one per line, from the tested registry. The parent factor below
# is DERIVED as (1 - sum), so adding a member here cannot silently break the partition.
DP_A  = CORRECTION_CFG['dp']    # 0.05  guided Viterbi on raw GR
VW_A  = CORRECTION_CFG['vw']    # 0.04  windowed decoder guided by v1warp
CAU_A = CORRECTION_CFG['cau']   # 0.02  causal decoder
P2_A  = CORRECTION_CFG['p2r']   # 0.05  re-decode of heavy
WL_A  = CORRECTION_CFG['wlvl']  # 0.03  wcorr3 windowed-level
WM_A  = CORRECTION_CFG['wmix']  # 0.03  wcorr3 windowed-mixture
DP_SMOOTH_W = 5   # v12: aggregated GR window for the dp emission only (1 = off; 0 raises)
assert DP_SMOOTH_W >= 1, 'DP_SMOOTH_W=0 builds an empty convolution kernel'
assert abs(sum([DP_A, VW_A, CAU_A, P2_A, WL_A, WM_A]) - 0.22) < 1e-12, \
    'correction members changed -- re-measure the OOF and update chassis/config.py'

def _wl_decode(g0, md_, anchor, Gt, Gg, grsc, v1, W=150, SUB=18, BAND=140, L=1500.0, GSC=20.0,
               LAM=6.0, MAXS=0.12, SLOPES=(-0.06, -0.03, -0.012, 0.0, 0.012, 0.03, 0.06)):
    from scipy.ndimage import gaussian_filter1d
    go = gaussian_filter1d(g0, 8)
    n = len(md_)
    idx = np.arange(W, n-W, SUB)
    if len(idx) < 3:
        return v1.copy(), v1.copy()
    TG = np.arange(anchor-BAND, anchor+BAND, 1.0); K = len(TG)
    mds = md_[idx]
    Ec = np.zeros((len(idx), K)); El = np.zeros((len(idx), K))
    for ii, i in enumerate(idx):
        obs = go[i-W:i+W]; dmd = md_[i-W:i+W]-md_[i]
        obs_z = (obs-obs.mean())/(obs.std()+1e-6)
        bc = np.full(K, -1.0); bl = np.full(K, 1e18)
        for s in SLOPES:
            tvt = TG[:, None]+s*dmd[None, :]
            ref = np.interp(tvt.ravel(), Gt, Gg).reshape(K, -1)
            rz = (ref-ref.mean(1, keepdims=True))/(ref.std(1, keepdims=True)+1e-6)
            bc = np.maximum(bc, (rz*obs_z[None, :]).mean(1))
            bl = np.minimum(bl, ((ref-obs[None, :])**2).mean(1))
        Ec[ii] = 1.0-bc; El[ii] = bl/grsc**2
    GU = (0.5*np.exp(-(mds-mds[0])/L))[:, None]*((TG[None, :]-v1[idx][:, None])/GSC)**2
    out = []
    for E in (El, 0.5*Ec/(np.median(Ec)+1e-9)+0.5*El/(np.median(El)+1e-9)):
        C = E/(np.median(E[E > 0])+1e-9) if (E > 0).any() else E
        C = C+GU
        V = C[0]+0.02*((TG-anchor)**2)/grsc
        ptr = np.zeros((len(idx), K), int)
        for i in range(1, len(idx)):
            dmd2 = max(mds[i]-mds[i-1], 1.0); ds = (TG[:, None]-TG[None, :])/dmd2
            T = V[None, :]+LAM*ds*ds; T[np.abs(ds) > MAXS] = 1e9
            j = np.argmin(T, 1); V = C[i]+T[np.arange(K), j]; ptr[i] = j
        p = np.zeros(len(idx), int); p[-1] = int(np.argmin(V))
        for i in range(len(idx)-1, 0, -1): p[i-1] = ptr[i, p[i]]
        out.append(np.interp(md_, mds, TG[p]))
    return out[0], out[1]

def _dp_decode(g0, md_, anchor, Gt, Gg, grsc, v1, W0=0.5, L=1500.0, GSC=20.0,
               BAND=140, SUB=18, LAM=6.0, MAXS=0.12, KIND='sq'):
    TG = np.arange(anchor-BAND, anchor+BAND, 1.0)
    Gc = np.interp(TG, Gt, Gg)
    idx = np.arange(0, len(md_), SUB)
    dd = (g0[idx][:, None]-Gc[None, :])/grsc
    E = np.log1p(dd*dd) if KIND == 'cau' else dd*dd
    nz = E[E > 0]
    C = E/(np.median(nz)+1e-9) if len(nz) else E
    mds = md_[idx]; v1s = v1[idx]
    C = C + (W0*np.exp(-(mds-mds[0])/L))[:, None]*((TG[None, :]-v1s[:, None])/GSC)**2
    V = C[0]+0.02*((TG-anchor)**2)/grsc; n, K = C.shape
    ptr = np.zeros((n, K), int)
    for i in range(1, n):
        dmd = max(mds[i]-mds[i-1], 1.0); ds = (TG[:, None]-TG[None, :])/dmd
        T = V[None, :]+LAM*ds*ds; T[np.abs(ds) > MAXS] = 1e9
        j = np.argmin(T, 1); V = C[i]+T[np.arange(K), j]; ptr[i] = j
    p = np.zeros(n, int); p[-1] = int(np.argmin(V))
    for i in range(n-1, 0, -1): p[i-1] = ptr[i, p[i]]
    return np.interp(md_, mds, TG[p])

stat("correction members", f"dp {DP_A} vw {VW_A} cau {CAU_A} p2r {P2_A} wlvl {WL_A} wmix {WM_A}")
stat("parent factor (derived)", f"{1.0-DP_A-VW_A-CAU_A-P2_A-WL_A-WM_A:.4f}")


## Stage G4 - run the decoders per well, then stage 4

Executes the decoders one well at a time and folds the six correction members in. The parent factor
is DERIVED as (1 - sum of members), so adding a member cannot silently break the partition of unity.
Every decoder falls back to the base prediction on failure, and `dp_fail` is printed: a well that
silently fell back would otherwise look like a healthy prediction.


In [ ]:
stage("G4", "decode per well, then stage 4")

dp_values = candidate_preoverlay.copy()          # fallback = base
vw_values = candidate_preoverlay.copy()
vwpp_values = candidate_preoverlay.copy()   # v19h: r_vw on smoothed GR
cau_values = candidate_preoverlay.copy()
wl_values = candidate_preoverlay.copy()
wm_values = candidate_preoverlay.copy()
dp_fail = []
for well in sorted(set(wellcol)):
    try:
        hw = pd.read_csv(COMP/'test'/f'{well}__horizontal_well.csv')
        tw = pd.read_csv(COMP/'test'/f'{well}__typewell.csv').sort_values('TVT')
        Gt = tw['TVT'].to_numpy(float); Gg = tw['GR'].to_numpy(float)
        okg = np.isfinite(Gg); Gt, Gg = Gt[okg], Gg[okg]
        ev, li = eval_rows(hw)
        md_ = hw['MD'].to_numpy(float)[ev]
        graw = hw['GR'].to_numpy(float)[ev]
        if not np.isfinite(graw).any() or len(ev) < 40 or len(Gt) < 40:
            raise ValueError('insufficient GR/typewell')
        g0 = pd.Series(graw).interpolate(limit_direction='both').to_numpy()
        anchor = float(hw['TVT_input'].iloc[li])
        grsc = 1.4826*np.median(np.abs(g0-np.median(g0)))+1e-6
        select = np.flatnonzero(wellcol == well)
        rid = ridx[select]
        # v1 on eval rows via csv-row-index alignment (same convention as the heavy series)
        v1_by_rid = pd.Series(public_parent[select], index=rid)
        v1w = v1_by_rid.reindex(ev).to_numpy(dtype=np.float64)
        okv = np.isfinite(v1w)
        if okv.mean() < 0.99: raise ValueError('v1/eval coverage mismatch')
        v1w = pd.Series(v1w).interpolate(limit_direction='both').to_numpy()
        vwg = pd.Series(candidate_preoverlay[select], index=rid).reindex(ev).to_numpy(dtype=np.float64)
        vwg = pd.Series(vwg).interpolate(limit_direction='both').to_numpy()
        gs = pd.Series(graw)
        g_roll = gs.fillna(gs.rolling(25, min_periods=3, center=True).median()).interpolate(limit_direction='both').to_numpy()
        gr_roll = 1.4826*np.median(np.abs(g_roll-np.median(g_roll)))+1e-6
        # DP EMISSION SMOOTHING (v12). The dp member alone reads an aggregated GR window;
        # vw and cau keep the series they were measured with. grsc stays RAW-derived --
        # the research harness scaled the emission by the raw MAD, so changing it here
        # would decode a different cost surface than the one that was validated.
        # Boxcar via convolve('same'), identical to research_directions/dp_rethink.smooth.
        g0_sm5 = np.convolve(g0, np.ones(DP_SMOOTH_W)/float(DP_SMOOTH_W), mode='same')
        assert g0_sm5.shape == g0.shape and np.isfinite(g0_sm5).all(), 'dp smoothing broke the GR series'
        for arr, gg, sc, gd, kk, ww in [(dp_values, g0_sm5, grsc, v1w, 'sq', 0.5),
                                        (vw_values, g0, grsc, vwg, 'sq', 0.5),
                                        (vwpp_values, g0_sm5, grsc, vwg, 'sq', 0.5),
                                        (cau_values, g_roll, gr_roll, v1w, 'cau', 0.3)]:
            dpv = _dp_decode(gg, md_, anchor, Gt, Gg, sc, gd, W0=ww, KIND=kk)
            back = pd.Series(dpv, index=ev).reindex(rid).to_numpy(dtype=np.float64)
            fin = np.isfinite(back)
            arr[select[fin]] = back[fin]
        wlv, wmv = _wl_decode(g0, md_, anchor, Gt, Gg, grsc, v1w)
        for arr, vals in ((wl_values, wlv), (wm_values, wmv)):
            back = pd.Series(vals, index=ev).reindex(rid).to_numpy(dtype=np.float64)
            fin = np.isfinite(back)
            arr[select[fin]] = back[fin]
    except Exception as err:
        dp_fail.append((well, type(err).__name__))
print('[DP] member computed; fallback wells:', dp_fail, flush=True)
p2_ok = np.isfinite(p2_values)
p2_filled = np.where(p2_ok, p2_values, candidate_preoverlay)
print("[P2] coverage:", float(p2_ok.mean()), flush=True)
candidate_preoverlay = (1.0-DP_A-VW_A-CAU_A-P2_A-WL_A-WM_A)*candidate_preoverlay + DP_A*dp_values + VW_A*vw_values + CAU_A*cau_values + P2_A*p2_filled + WL_A*wl_values + WM_A*wm_values

apply_leak = g_maek['apply_leak']

vec("candidate (stage 4)", candidate_preoverlay)

# ---- failure budget: guided decoders ------------------------------------------------
# A failed well keeps every correction member equal to the base, so the 0.22 correction mass
# silently becomes the identity there. Same asymmetry as above: warn, fail only on collapse.
_n_dec = len(set(wellcol))
_dec_frac = len(dp_fail) / max(_n_dec, 1)
if dp_fail:
    warn(f"decoders fell back on {len(dp_fail)}/{_n_dec} wells ({100 * _dec_frac:.1f}%)")
else:
    ok(f"decoders ran on all {_n_dec} wells")
assert _dec_frac <= 0.20, (
    f"FATAL: guided decoders failed on {len(dp_fail)}/{_n_dec} wells -- the 0.22 correction "
    f"mass is the identity for most of the field")
_heavy_nan = int((~np.isfinite(heavy_values)).sum())
if _heavy_nan:
    warn(f"{_heavy_nan:,}/{len(heavy_values):,} rows have no STRIDE heavy -> parent is maek-only there")


## Stage G4b - dp_rate_ens

The one member with no guide: a rate-space DP with Bresenham remainder-carry displacements,
averaged over KAPPA {0.25, 0.5, 1.0}. Parent-free, so it is deterministic per well and was
verified against the bank to one float32 ULP. Wells that fail fall back to the anchor,
which is the convention the banked member used.


In [ ]:
# ============ v19h: dp_rate_ens member (parent-free rate-space DP) ============
# dp_rate2.py recipe verbatim: remainder-carry (Bresenham) displacements so a held
# sub-grid rate is representable; ensemble mean over KAPPA {0.25, 0.5, 1.0}; NO guide
# (parent-free) -- deterministic per well, verified vs the bank to 1 float32 ULP.
# Failing/short wells fall back to the anchor (the banked convention).
stage("G4b", "dp_rate_ens (parent-free rate DP, kappa ensemble)")
_B_BAND = 140

def _dp_rate_carry(g0, md_, anchor, Gt, Gg, grsc, ir=0.0,
                   RMAX=0.12, NR=25, KAPPA=0.5, RJUMP=2):
    TG = np.arange(anchor - _B_BAND, anchor + _B_BAND, 1.0)
    Gc = np.interp(TG, Gt, Gg)
    RG = np.linspace(-RMAX, RMAX, NR)
    idx = np.arange(0, len(md_), 18)
    mds = md_[idx]
    dd = (g0[idx][:, None] - Gc[None, :]) / grsc
    E = dd * dd
    nz = E[E > 0]
    C = E / (np.median(nz) + 1e-9) if len(nz) else E
    n, K = C.shape
    NRr = len(RG)
    BIG = 1e18
    CUM = np.rint(RG[None, :] * (mds[:, None] - mds[0])).astype(int)
    SH = np.diff(CUM, axis=0)
    V = np.full((K, NRr), BIG)
    p0 = np.argmin(np.abs(TG - anchor))
    for r in range(NRr):
        V[p0, r] = C[0, p0] + 0.5 * ((RG[r] - ir) / max(RMAX, 1e-9) * 3.0) ** 2
    ptr_p = np.zeros((n, K, NRr), np.int16)
    ptr_r = np.zeros((n, K, NRr), np.int16)
    for i in range(1, n):
        Vn = np.full((K, NRr), BIG)
        for dr in range(-RJUMP, RJUMP + 1):
            cost_r = KAPPA * (dr ** 2)
            for rn in range(NRr):
                rp = rn - dr
                if rp < 0 or rp >= NRr:
                    continue
                sh = int(SH[i - 1, rn])
                src = V[:, rp]
                cand = np.full(K, BIG)
                if sh == 0:
                    cand = src.copy()
                elif sh > 0:
                    cand[sh:] = src[:-sh]
                else:
                    cand[:sh] = src[-sh:]
                tot = cand + cost_r
                upd = tot < Vn[:, rn]
                Vn[upd, rn] = tot[upd]
                ptr_r[i, upd, rn] = rp
                pp = np.arange(K) - sh
                ptr_p[i, upd, rn] = np.clip(pp[upd], 0, K - 1)
        V = Vn + C[i][:, None]
    fp, fr = np.unravel_index(int(np.argmin(V)), V.shape)
    path = np.zeros(n, int)
    path[-1] = fp
    cp, cr = fp, fr
    for i in range(n - 1, 0, -1):
        np_, nr_ = int(ptr_p[i, cp, cr]), int(ptr_r[i, cp, cr])
        path[i - 1] = np_
        cp, cr = np_, nr_
    return np.interp(md_, mds, TG[path])



## Stage G4b (run) - decode dp_rate_ens per well

Deterministic per well: no guide, no fitted parameters. A well that fails is pinned to its
anchor, which is the convention the banked member used, and the failure budget below makes
that visible rather than silent.


In [ ]:
dpre_values = candidate_preoverlay.copy()   # fallback = base (never fails on hidden)
_dpre_stats = {"ok": 0, "fill": 0}
for _well in sorted(set(wellcol)):
    _sel = np.flatnonzero(wellcol == _well)
    try:
        _hw = pd.read_csv(COMP / 'test' / f'{_well}__horizontal_well.csv')
        _tw = pd.read_csv(COMP / 'test' / f'{_well}__typewell.csv').sort_values("TVT")
        _Gt = _tw["TVT"].to_numpy(float)
        _gg = _tw["GR"].astype(float).interpolate(limit_direction="both")
        _Gg = _gg.fillna(_gg.mean()).to_numpy(float)
        _ev, _ = eval_rows(_hw)
        _kn = _hw[_hw["TVT_input"].notna()]   # notna-FILTERED: NaNs before the anchor
                                              # would poison _tail's diff if we sliced
        _md = _hw["MD"].to_numpy(float)[_ev]
        _graw = _hw["GR"].to_numpy(float)[_ev]
        _g0 = pd.Series(_graw).interpolate(limit_direction="both").to_numpy()
        if not np.isfinite(_g0).all():
            _g0 = np.nan_to_num(_g0, nan=float(np.nanmedian(_Gg)))
        assert len(_kn) >= 10 and len(_md) >= 200 and len(_md) == len(_sel), \
            f"dp_rate_ens row mismatch {_well}: {len(_md)} vs {len(_sel)}"
        _anchor = float(_kn["TVT_input"].iloc[-1])
        _grsc = 1.4826 * np.median(np.abs(_g0 - np.median(_g0))) + 1e-6
        _tail = _kn.tail(30)
        _dt = np.diff(_tail["TVT_input"].to_numpy(float))
        _dm = np.diff(_tail["MD"].to_numpy(float))
        _m = _dm > 0
        _ir = float(np.median(_dt[_m] / _dm[_m])) if _m.sum() >= 3 else 0.0
        _preds = [_dp_rate_carry(_g0, _md, _anchor, _Gt, _Gg, _grsc, ir=_ir, KAPPA=_k)
                  for _k in (0.25, 0.5, 1.0)]
        dpre_values[_sel] = np.mean(_preds, axis=0)
        _dpre_stats["ok"] += 1
    except Exception as _e:
        _anchor_fill = float(pd.read_csv(COMP / 'test' / f'{_well}__horizontal_well.csv')
                             ["TVT_input"].dropna().iloc[-1])
        dpre_values[_sel] = _anchor_fill
        _dpre_stats["fill"] += 1
        print(f"[DPRE] {_well} fell back to anchor: {type(_e).__name__} {_e}", flush=True)
print(f"[DPRE] decoded {_dpre_stats['ok']} wells, anchor-filled {_dpre_stats['fill']}",
      flush=True)
# ---- failure budget: dp_rate_ens ----------------------------------------------------
# 5-pass review: every other member had one (STRIDE and the guided decoders at <=20%,
# dcorr and uproj at >=90% of wells, mixpf at >99.9% rows) -- this loop only printed.
# A well that falls back here is pinned to its ANCHOR, so at w=0.0192 a total collapse
# would move the blend by ~0.0192*|blend-anchor| ~ 0.2 ft: bounded, but it should not be
# the one member that can degrade silently. Same asymmetry as the others: warn on any
# fallback, fail only on collapse.
_dpre_frac = _dpre_stats["fill"] / max(len(set(wellcol)), 1)
if _dpre_stats["fill"]:
    warn(f"dp_rate_ens anchor-filled {_dpre_stats['fill']}/{len(set(wellcol))} wells "
         f"({100 * _dpre_frac:.1f}%)")
else:
    ok(f"dp_rate_ens decoded all {_dpre_stats['ok']} wells")
assert _dpre_frac <= 0.20, (
    f"FATAL: dp_rate_ens fell back to the anchor on {_dpre_stats['fill']}/"
    f"{len(set(wellcol))} wells -- the combo-B member is anchor-flat for most of the field")


## Stage G5a - newfeats and combo B

Stage 4b adds newfeats as an independent member, then combo B folds in dp_rate_ens and
r_vw_pp. The gain, the capped dcorr, the U-space smoother and the overlay each have their
own cell below -- this header used to describe all of them, from when G5 was one 225-line
cell.


In [ ]:
stage("G5", "stage 4b, gain, overlay, artifacts")

# ---- amplitude gain: correct ~9% ensemble shrinkage toward per-well anchor (nested-CV G=1.08, 7/8 spatial blocks; OOF 7.1989->7.101, transfer 6.7786->6.606) ----
# ================= STAGE 4b: newfeats as an INDEPENDENT member (v5) =================
# The production WARP keeps its 0.20 slot at stage 3; newfeats enters HERE, beside the
# whole blend, scaling everything else by (1 - w). Nested GroupKFold(5) by well,
# 3,783,582 rows / 772 wells, reference 7.1000 @ G=1.08:
#     SWAP slot -> newfeats (v3/v4 shipped) .... 7.0445  (-0.0565)
#     ADD newfeats here, w=0.16, G=1.12 ........ 6.9633  (-0.1377)  <- this
# Per-fold picks were w = [.16 .16 .16 .16 .16], G = [1.11 1.13 1.11 1.13 1.12]: flat in w,
# so the frozen (0.16, 1.12) is the fold-consensus rather than a tuned point.
# RE-DERIVED 2026-07-25 from oof_bank/ (all 9 members verified against their published
# standalones; reference 7.1011 @ G=1.08 reproduces exactly). The earlier figures here
# (7.0467 / 6.9726) came from a pass that OMITTED p2r. NOTE the SWAP was previously
# published as 7.0528 because it had been scored at the ADD gain (~1.125) instead of its
# own optimum 1.09; corrected, ADD's margin over SWAP is 0.081, not 0.089.
NEWFEATS_W = NEWFEATS_W_CFG              # 0.16
assert 0.0 <= NEWFEATS_W <= 0.40, NEWFEATS_W
# REVIEW FIX (v31): the old print measured |newfeats - candidate| AFTER the update, i.e.
# (1-w)*|newfeats - before|, which overstated the step by (1-w)/w = 5.25x. Report the move
# this stage actually made.
_nf_before = candidate_preoverlay.copy()
candidate_preoverlay = (1.0 - NEWFEATS_W) * candidate_preoverlay + NEWFEATS_W * newfeats_values
print('[NEWFEATS] added as member w=%.2f, mean|move|=%.4f ft, max %.4f ft'
      % (NEWFEATS_W, float(np.mean(np.abs(candidate_preoverlay - _nf_before))),
         float(np.max(np.abs(candidate_preoverlay - _nf_before)))), flush=True)

# ===== v19h combination B: two broad-improvement members (dual-objective search) =====
# Gate record (nested 5x5 vs ms base 6.8035): B = 6.7780 (-0.0255), fold-wins 80%,
# improvement across ALL five difficulty quintiles (55/51/54/55/53%), created damage 0
# wells; both members are correction-class adds, the only class with a clean LB-transfer
# record (v12 dp corrector: 72% wins -> transferred -0.017).
B_DPRE_W, B_VWPP_W = 0.0192, 0.04
candidate_preoverlay = ((1.0 - B_DPRE_W - B_VWPP_W) * candidate_preoverlay
                        + B_DPRE_W * dpre_values + B_VWPP_W * vwpp_values)
print('[COMBO-B] dp_rate_ens w=%.4f + r_vw_pp w=%.4f applied' % (B_DPRE_W, B_VWPP_W),
      flush=True)
pd.DataFrame({'id': ids, 'well': wellcol, 'dp_rate_ens': dpre_values,
              'r_vw_pp': vwpp_values}).to_parquet('/kaggle/working/b_members.parquet')



## Stage G5b - amplitude gain and the capped dcorr injection

The gain corrects the ~9% shrinkage a convex blend induces toward the per-well anchor. The
dcorr term is then injected at the DERIVED effective st_heavy weight, bounded by `DCORR_CAP`.


In [ ]:
GAIN = GAIN_CFG                          # 1.12  calibration; refit whenever contributions change
_anchor_row = np.full(len(ids), np.nan, dtype=np.float64)
for _well in sorted(set(wellcol)):
    _hw = pd.read_csv(COMP/'test'/f'{_well}__horizontal_well.csv')
    _kn = _hw['TVT_input'].notna().to_numpy()
    _a = float(_hw['TVT_input'].to_numpy(float)[np.flatnonzero(_kn)[-1]])
    _anchor_row[wellcol == _well] = _a
assert np.isfinite(_anchor_row).all(), 'anchor coverage failure'
# REVIEW FIX (v31): the old print contained `- _anchor_row + _anchor_row - _anchor_row`,
# which collapses to a formula in anchor/GAIN - 2*anchor and logged 153.328 ft for a 12%
# gain on a ~10 ft excursion. The displacement of this stage is (GAIN-1)*|pre - anchor|.
_gain_before = candidate_preoverlay.copy()
candidate_preoverlay = _anchor_row + GAIN * (candidate_preoverlay - _anchor_row)
print('[GAIN] applied G=%.2f, mean|move|=%.4f ft, max %.4f ft'
      % (GAIN, float(np.mean(np.abs(candidate_preoverlay - _gain_before))),
         float(np.max(np.abs(candidate_preoverlay - _gain_before)))), flush=True)

# ===== v20: U-space robust polynomial projection (from the public-notebook sweep) =====
# Per well, fit U = TVT + Z against normalised MD with a degree-6 polynomial under IRLS
# reweighting, then move the prediction 30% of the way to that fit. This is a pure
# post-processor on our own prediction -- it reads no truth and no train-side file.
# Gate on our bank vs the v19h blend: nested -0.0156, 100% fold wins, majority improvement
# in ALL five difficulty quintiles, ZERO wells damaged >2 ft, bootstrap 7.4%@39 / 0.2%@151.
# ===== v22b: add the capped dcorr contribution to the blend ======================
# Algebraically identical to swapping the member inside the stage-3 blend (verified
# against the bank to 1.8e-12), which is the configuration the gate scored:
#   nested -0.1199, 100% fold wins, ZERO wells damaged >2 ft (max +1.30 ft),
#   majority improvement in ALL five difficulty quintiles, P(worse) 12.6%@39.
# The uncapped swap is -0.3013 nested but damages 14 wells by up to 5.2 ft -- that is
# the configuration that lost 0.484 on the public LB as v19a.
# REVIEW FIX (v31): 0.1572 was hand-typed here. It is the EFFECTIVE weight st_heavy
# carries into the final blend, and the notebook's own rule is that such weights are
# derived, never written out -- a hand-typed copy goes silently stale the moment any
# upstream weight moves, and nothing asserted it. Derived below and cross-checked against
# the historical literal, which is the value every prior submission used.
_HEAVY_EFF = (PARENT_HEAVY * (1.0 - WARP_REUSE_WEIGHT)
              * (1.0 - DP_A - VW_A - CAU_A - P2_A - WL_A - WM_A) * (1.0 - NEWFEATS_W))
assert abs(_HEAVY_EFF - 0.1572) < 5e-4, (
    f'effective st_heavy weight moved to {_HEAVY_EFF:.6f}; the shipped chassis assumed '
    '0.1572. An upstream weight changed -- re-derive the dcorr coefficient deliberately.')
_DC_COEF = GAIN * (1.0 - B_DPRE_W - B_VWPP_W) * _HEAVY_EFF
print('[DCORR] effective st_heavy weight %.6f (was hand-typed 0.1572), coef %.6f'
      % (_HEAVY_EFF, _DC_COEF), flush=True)
_dc_before = candidate_preoverlay.copy()
candidate_preoverlay = candidate_preoverlay + _DC_COEF * dcorr_delta
print('[DCORR] injected coef %.6f, mean|move| %.4f ft, max %.4f ft'
      % (_DC_COEF, float(np.mean(np.abs(candidate_preoverlay - _dc_before))),
         float(np.max(np.abs(candidate_preoverlay - _dc_before)))), flush=True)
assert float(np.max(np.abs(candidate_preoverlay - _dc_before))) <= _DC_COEF * DCORR_CAP + 1e-6, 'dcorr injection exceeded its own bound'



In [ ]:
# ===================== v34a: nn_warpdrop injected as an independent member =====================
# WHERE: after GAIN and the (disabled) dcorr injection, BEFORE uproj2. This is exactly the point
# the offline analysis measured -- raw = anchor + GAIN*(...), then (1-w)*raw + w*nn, then project.
# Injecting after uproj2 or before GAIN would measure a different quantity.
#
# WHY IT EARNS A SLOT: standalone 9.67 ft, worse than several nets already in the blend, but its
# errors correlate 0.600 with the blend against newfeats' 0.681 -- the most decorrelated net in
# the bank. Offline: v21 6.7724 -> 6.7237 at w=0.06, zero wells damaged beyond 2 ft.
NN_W = 0.06

import keras as _keras
from keras import ops as _kops

# ---- 1. the coordinate transform must be the one the weights were fitted under ----------------
# The trainer floors the variance at 1e-9; this chassis floors it at 1.0. For coordinates spanning
# thousands of feet both reduce to the raw variance, but assert it rather than assume.
_cm_tr = coord_sum / coord_n
_cs_tr = np.sqrt(np.maximum(coord_sum2 / coord_n - _cm_tr ** 2, 1e-9))
assert np.allclose(_cm_tr, coord_mean, rtol=0, atol=1e-9), 'coord_mean differs from training'
assert np.allclose(_cs_tr, coord_std,  rtol=1e-12), (
    f'coord_std differs from training: {_cs_tr} vs {coord_std} -- the variance floor bit')
print('[NN] coordinate normalisation matches the trainer', flush=True)

# ---- 2. make_sample in its own namespace, same pattern the WARP nets use ----------------------
_nns = {}
exec(compile(WARP_SOURCE, '/kaggle/working/_nn_costvolume.py', 'exec'), _nns)
_nns['coord_mean'], _nns['coord_std'] = coord_mean, coord_std
_nn_make_sample = _nns['make_sample']

# ---- 3. the network, ported verbatim from the trainer ----------------------------------------
# Any structural difference makes load_weights fail loudly or, far worse, load silently wrong.
_NF = 33
def _nn_build(ntw, ntwc):
    hin = _keras.Input((None, _NF)); din = _keras.Input((None,))
    ain = _keras.Input((1,));        tin = _keras.Input((ntw, ntwc))
    t = _keras.layers.Conv1D(48, 5, padding='same', activation='gelu')(tin)
    t = _keras.layers.Conv1D(48, 3, padding='same', dilation_rate=2, activation='gelu')(t)
    t = _keras.layers.GlobalAveragePooling1D()(t)
    tb = _keras.layers.Lambda(lambda q: _kops.expand_dims(q[0], 1) + 0 * _kops.expand_dims(q[1], -1),
                              output_shape=(None, 48))([t, din])
    dex = _keras.layers.Lambda(lambda q: _kops.expand_dims(q, -1), output_shape=(None, 1))(din)
    x = _keras.layers.Concatenate()([hin, dex, tb])
    h = _keras.layers.Conv1D(80, 5, padding='same', activation='gelu')(x)
    for _dl in (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024):      # receptive field 4098 rows
        r = _keras.layers.Conv1D(80, 3, padding='same', dilation_rate=_dl, activation='gelu')(h)
        r = _keras.layers.Dropout(0.10)(r)
        h = _keras.layers.LayerNormalization()(_keras.layers.Add()([h, r]))
    f = _keras.layers.Conv1D(96, 1, activation='gelu')(h)
    raw = _keras.layers.Conv1D(1, 1, kernel_initializer='zeros', bias_initializer='zeros')(f)
    traj = _keras.layers.Lambda(
        lambda q: _kops.expand_dims(q[2], -1) + 120.0 * _kops.tanh(
            _kops.cumsum(0.08 * _kops.tanh(q[0]) * _kops.expand_dims(q[1], -1), axis=1) / 120.0),
        output_shape=(None, 1))([raw, din, ain])
    return _keras.Model([hin, din, ain, tin], traj)

# ---- 4. build one sample per test well --------------------------------------------------------
_nn_samples, _nn_ntw, _nn_ntwc = [], None, None
for _path in test_files:
    _w = _path.name.replace('__horizontal_well.csv', '')
    _hw = pd.read_csv(_path).reset_index(drop=True)
    # the trainer sorts the typewell by TVT; every shipped file already is, but match it exactly
    _tw = pd.read_csv(_path.parent / f'{_w}__typewell.csv').sort_values('TVT')
    _ev = np.flatnonzero(_hw['TVT_input'].isna().to_numpy())
    _bnd = int(_ev[0]) - 1
    _fake = _hw.copy()
    _anc = float(_fake['TVT_input'].iat[_bnd])
    _fake['TVT'] = _fake['TVT_input'].fillna(_anc)
    _s = _nn_make_sample({'well': _w, 'hw': _fake, 'tw': _tw}, _bnd,
                         real_boundary=True, allow_short=True)
    assert _s is not None, f'[NN] make_sample returned None for {_w}'
    assert _s['h'].shape[1] == _NF, f"[NN] feature width {_s['h'].shape[1]} != {_NF} for {_w}"
    if _nn_ntw is None:
        # trainer: NTW = shape[0]; NTWC = shape[1] if len(shape) > 1 else 1, then reshape
        _tsh = _s['tw'].shape
        _nn_ntw, _nn_ntwc = _tsh[0], (_tsh[1] if len(_tsh) > 1 else 1)
    assert _s['tw'].shape[0] == _nn_ntw and _s['tw'].size == _nn_ntw * _nn_ntwc, \
        f'[NN] typewell shape drift on {_w}: {_s["tw"].shape}'
    if len(_s['h']) > 10240:
        warn(f"[NN] {_w} has {len(_s['h'])} rows, beyond the trainer's 10240 cap")
    _nn_samples.append({'well': _w, 'n_ev': len(_ev), 's': _s})
print(f'[NN] built {len(_nn_samples)} samples; typewell {_nn_ntw}x{_nn_ntwc}, '
      f'longest {max(len(q["s"]["h"]) for q in _nn_samples)} rows', flush=True)

# ---- 5. five folds, averaged (the same combination the chassis uses for the WARP nets) ---------
_nn_net = _nn_build(_nn_ntw, _nn_ntwc)
print(f'[NN] parameters {_nn_net.count_params():,}', flush=True)
_nn_paths = sorted(glob.glob('/kaggle/input/**/warpdrop_seed_f[0-4].weights.h5',
                             recursive=True))
assert len(_nn_paths) == 5, f'[NN] expected 5 fold weights, found {len(_nn_paths)}: {_nn_paths}'
_nn_hashes = {Path(p).name: hashlib.sha256(open(p, 'rb').read()).hexdigest() for p in _nn_paths}
_nn_acc = {q['well']: np.zeros(len(q['s']['h']), np.float64) for q in _nn_samples}
for _wp in _nn_paths:
    _nn_net.load_weights(_wp)
    for _q in _nn_samples:
        _s = _q['s']
        # same call pattern the chassis uses for the WARP nets -- direct call, not
        # predict_on_batch, which adds its own tracing layer for no benefit at batch 1
        _out = _nn_net((
            tf.convert_to_tensor(_s['h'][None],   tf.float32),
            tf.convert_to_tensor(_s['dmd'][None], tf.float32),
            tf.convert_to_tensor([[_s['anchor_tvt']]], tf.float32),
            tf.convert_to_tensor(_s['tw'].reshape(1, _nn_ntw, _nn_ntwc), tf.float32),
        ), training=False).numpy()
        _nn_acc[_q['well']] += np.asarray(_out[0, :, 0], np.float64)
    print(f'[NN] fold done: {Path(_wp).name}', flush=True)
for _k in _nn_acc:
    _nn_acc[_k] /= len(_nn_paths)
del _nn_net
tf.keras.backend.clear_session()
import gc as _gc; _gc.collect()

# ---- 6. map onto the submission rows ----------------------------------------------------------
nn_values = np.full(len(ids), np.nan, dtype=np.float64)
for _q in _nn_samples:
    _sel = np.flatnonzero(wellcol == _q['well'])
    _v = _nn_acc[_q['well']]
    assert len(_sel) == len(_v), (f"[NN] row mismatch on {_q['well']}: "
                                  f'{len(_sel)} submission rows vs {len(_v)} predictions')
    nn_values[_sel] = _v
assert np.isfinite(nn_values).all(), '[NN] coverage/finite failure'
_nn_std = float(np.std(nn_values - _anchor_row))
print('[NN] pred std about the anchor %.2f ft (trainer reported 13-15 ft per fold)' % _nn_std,
      flush=True)
assert 5.0 < _nn_std < 40.0, (
    f'[NN] prediction spread {_nn_std:.2f} ft is implausible -- a collapsed net that simply '
    'returns the anchor would pass every other check here')
vec('nn_warpdrop', nn_values)

# ---- 7. inject --------------------------------------------------------------------------------
_nn_before = candidate_preoverlay.copy()
candidate_preoverlay = (1.0 - NN_W) * candidate_preoverlay + NN_W * nn_values
print('[NN] injected w=%.3f, mean|move| %.4f ft, max %.4f ft'
      % (NN_W, float(np.mean(np.abs(candidate_preoverlay - _nn_before))),
         float(np.max(np.abs(candidate_preoverlay - _nn_before)))), flush=True)
assert float(np.max(np.abs(candidate_preoverlay - _nn_before))) > 1e-6, \
    '[NN] injection was a silent no-op'





In [ ]:
# ================================================================ ROW-WISE DISAGREEMENT GAIN
# SLOT A (conservative): IQR statistic, cap [0.95, 1.03].
# v35a. Rescales the blend's drift by a gain keyed to ROW-LEVEL MEMBER DISAGREEMENT: where the
# 11 members agree the blend is trusted and moved slightly less; where they diverge it is moved
# slightly more. Piecewise constant on spread deciles, frozen from all 772 train wells.
#
# WHERE: after the NN injection, BEFORE uproj2 -- the exact point the offline measurement used
# (research_directions/gain_audit2.py / _audit3.py). Nested GroupKFold(5) by well:
#   NN alone (v34a)          -0.0476
#   + this gain [0.95,1.03]  -0.0801   win 56.1%  median adv -0.0566  0 wells damaged >2 ft
#                                      bootstrap 81.6% @39, 97.1% @151
# Additive with the NN rather than redundant (-0.0476 + -0.0462 ~ -0.0946 at the wider cap).
# The cap is ASYMMETRIC on purpose: the amplify branch adds pooled gain with no distributional
# improvement (tail-shaped), the shrink branch broadens the win, so the risky side is capped
# harder. Stable across 6 fold partitions (+/-0.0036) and flat in bin count from 4 to 40.
_GAIN_EDGES = np.array([1.2231, 1.9349, 2.6061, 3.2726, 3.9634, 4.7295, 5.6348, 6.7168, 8.0682, 10.0356, 13.5471])
_GAIN_VALS  = np.array([0.91876, 0.93764, 0.93163, 0.94215, 0.96365, 0.96268, 0.96118, 0.97428, 0.98862, 0.98248, 1.02745, 1.17911])
_GAIN_LO, _GAIN_HI = 0.95, 1.03
_TRAIN_SPREAD_Q = [0.402, 0.903, 2.606, 4.729, 8.068, 16.125, 27.946]   # 1/5/25/50/75/95/99

# Row-wise std ACROSS members. The anchor cancels in a row-wise std, so it never enters here
# (verified against the anchor-subtracted form on the bank).
_GM = np.column_stack([maek_values, heavy_values, warp_values, newfeats_values, mixpf_values,
                       p2_filled, dpre_values, dp_values, vwpp_values, cau_values, nn_values])
assert _GM.shape[1] == 11, _GM.shape
_nfin = np.isfinite(_GM).sum(axis=1)
print('[GAINMAP] members %s   rows with all 11 finite %.4f   min finite %d'
      % (_GM.shape, float((_nfin == 11).mean()), int(_nfin.min())), flush=True)
# A std over fewer members is biased LOW and would land the row in the wrong bin, so rows with
# thin member coverage are left untouched (g=1) rather than mis-binned. Killing a multi-hour run
# outright here would be worse than degrading gracefully, so the hard check is on the FRACTION.
_thin = _nfin < 10
print('[GAINMAP] rows left untouched for thin member coverage: %d (%.5f)'
      % (int(_thin.sum()), float(_thin.mean())), flush=True)
assert float(_thin.mean()) < 0.05, 'over 5%% of rows have thin member coverage; map not applicable'

with np.errstate(invalid='ignore'):
    _spread = (np.nanpercentile(_GM, 75, axis=1) - np.nanpercentile(_GM, 25, axis=1))
_spread = np.where(np.isfinite(_spread), _spread, 0.0)

# Distribution check: the map is keyed to absolute spread, so a shifted test-side spread
# distribution would silently apply the wrong bins. Report it rather than assume it matches.
_q = np.percentile(_spread, [1, 5, 25, 50, 75, 95, 99])
print('[GAINMAP] spread q1/5/25/50/75/95/99 test %s' % np.round(_q, 3).tolist(), flush=True)
print('[GAINMAP]                            train %s' % _TRAIN_SPREAD_Q, flush=True)
print('[GAINMAP] median ratio test/train %.3f' % (_q[3] / _TRAIN_SPREAD_Q[3]), flush=True)

# BIN BY WITHIN-RUN RANK, NOT BY THE FROZEN ABSOLUTE EDGES.
# The bank's members are single-model OOF; deployment averages 5 folds, which compresses extreme
# disagreement. Measured on the 3 pipeline-test wells, which are also in train: spread matches the
# bank through the median (0.93/1.46/2.54/3.83 vs 1.01/1.44/2.59/3.75) but the tail collapses
# (q99 8.6 vs 13.0). Frozen absolute edges therefore under-fill the top bins at deployment -- the
# first run put 99.3% of rows in the shrink branch and only 0.7% in amplify, against 1/12 expected.
# The map was fitted on DECILES, so binning by rank applies it exactly as fitted and is immune to
# any global scale shift. Offline this is free: -0.0793 rank vs -0.0801 absolute, same win rate
# (55.8% vs 56.1%), same zero damage, same bootstrap (96.5% vs 96.6% @151).
_rank_edges = np.quantile(_spread[~_thin], np.linspace(0, 1, len(_GAIN_VALS) + 1)[1:-1])
print('[GAINMAP] within-run rank edges %s' % np.round(_rank_edges, 3).tolist(), flush=True)
print('[GAINMAP] frozen train edges    %s' % np.round(_GAIN_EDGES, 3).tolist(), flush=True)
_g = np.clip(_GAIN_VALS[np.searchsorted(_rank_edges, _spread)], _GAIN_LO, _GAIN_HI)
_g = np.where(_thin, 1.0, _g)          # thin-coverage rows pass through unchanged

# RECENTRE so the map does NOT touch the global scale.
# The chassis already applies GAIN=1.12; multiplying the drift again means any drift of the map's
# average multiplier away from 1 silently re-fits that global gain. Measured: the raw map has a
# dd-weighted mean of 0.99126, pulling the effective gain 1.12 -> 1.1102 -- and that is the WRONG
# direction, because the optimal extra global multiplier is 1.0202, i.e. 1.12 is already slightly
# under-scaled. The map was compounding an existing miscalibration instead of adding information.
# Forcing the dd-weighted mean to exactly 1 makes the map purely RELATIVE (redistribute step size
# across rows by competence) and leaves GAIN exactly as calibrated. It is also free:
#   as pushed  -0.0730   win 54.7%  b151 94.4%
#   recentred  -0.0799   win 55.2%  b151 95.6%      (-0.0856 +/- 0.0040 over 4 fold partitions)
# Weights are the squared drift, which is what the pooled metric actually feels, and the mean is
# taken WITHIN THE RUN so the result is net-neutral on whatever wells are actually scored.
# Deliberately NOT re-clipped afterwards: re-clipping would reintroduce the very shift this
# removes. The rescale is tiny, so the effective range stays bounded (train: 0.959..1.040).
_drift_now = candidate_preoverlay - _anchor_row
_w2 = _drift_now * _drift_now
_gmean = float((_g * _w2).sum() / max(_w2.sum(), 1e-12))
print('[GAINMAP] dd-weighted mean multiplier before recentring %.5f -> effective gain %.5f'
      % (_gmean, 1.12 * _gmean), flush=True)
_g = _g / _gmean
print('[GAINMAP] recentred: dd-weighted mean %.6f   g range [%.4f, %.4f]'
      % (float((_g * _w2).sum() / max(_w2.sum(), 1e-12)), float(_g.min()), float(_g.max())),
      flush=True)
assert _GAIN_LO - 0.04 < float(_g.min()) and float(_g.max()) < _GAIN_HI + 0.04, \
    'recentred gain left its sane range'
_gm_before = candidate_preoverlay.copy()
candidate_preoverlay = _anchor_row + _g * (candidate_preoverlay - _anchor_row)
_mv = np.abs(candidate_preoverlay - _gm_before)
print('[GAINMAP] applied cap [%.2f,%.2f]  mean|move| %.4f ft  max %.4f ft  '
      'rows shrunk %.3f  amplified %.3f'
      % (_GAIN_LO, _GAIN_HI, float(_mv.mean()), float(_mv.max()),
         float((_g < 1).mean()), float((_g > 1).mean())), flush=True)
assert float(_mv.max()) > 1e-6, 'gain map was a no-op -- it did not reach the prediction'
assert float(_mv.max()) < 60.0, 'gain map moved a row implausibly far'
vec("v35a gainmap", candidate_preoverlay)








## Stage G5c - U-space smoother

Per well, fit `U = TVT + Z` against normalised MD with a degree-6 IRLS polynomial and move
30% of the way to it, after a gaussian low-pass. A pure post-processor: it reads no truth
and no train-side file. Wells whose fit diverges are skipped rather than asserted.


In [ ]:
from scipy.ndimage import gaussian_filter1d as _pj_g1d
_PJ_DEG, _PJ_W, _PJ_IRLS = 6, 0.30, 4
_PJ_GSIG, _PJ_GW = 80.0, 0.80     # local stage: sigma in ROWS (~1 ft/row), weight
_pj_done, _pj_maxmove = 0, 0.0
for _well in sorted(set(wellcol)):
    _sel = np.flatnonzero(wellcol == _well)
    _hw = pd.read_csv(COMP / 'test' / f'{_well}__horizontal_well.csv')
    _ev, _ = eval_rows(_hw)
    _md = _hw['MD'].to_numpy(float)[_ev]
    _Zc = _hw['Z'].to_numpy(float)[_ev]
    if len(_md) != len(_sel) or len(_md) < 8 * (_PJ_DEG + 1):
        continue                      # row-count mismatch or too short to fit safely
    _span = float(_md[-1] - _md[0])
    if not np.isfinite(_span) or _span <= 1.0:
        continue
    _s = (_md - _md[0]) / _span
    _u_raw = candidate_preoverlay[_sel] + _Zc
    # stage 1 (local): gaussian low-pass removes prediction wiggle the truth does not have
    _u = (1.0 - _PJ_GW) * _u_raw + _PJ_GW * _pj_g1d(_u_raw, _PJ_GSIG, mode='nearest')
    _u0 = _u - _u[0]
    if not np.isfinite(_u0).all() or not np.isfinite(_u).all():
        continue
    try:
        _c = np.polyfit(_s, _u0, _PJ_DEG)
        for _ in range(_PJ_IRLS):
            _r = _u0 - np.polyval(_c, _s)
            _sc = 1.4826 * np.median(np.abs(_r - np.median(_r))) + 1e-9
            _c = np.polyfit(_s, _u0, _PJ_DEG,
                            w=np.sqrt(1.0 / (1.0 + (_r / (2.0 * _sc)) ** 2)))
        _fit = np.polyval(_c, _s) + _u[0]
    except Exception as _e:
        print(f'[UPROJ] {_well} fit failed ({type(_e).__name__}); left unchanged', flush=True)
        continue
    if not np.isfinite(_fit).all():
        continue
    # stage 2 (global): move toward the robust polynomial shape
    _new = ((1.0 - _PJ_W) * _u + _PJ_W * _fit) - _Zc
    # A smoother must never take the run down: an implausible move means the fit diverged
    # on this well, so drop the well rather than assert (the base prediction stays).
    _mx = float(np.nanmax(np.abs(_new - candidate_preoverlay[_sel])))
    if not np.isfinite(_mx) or _mx > 60.0:
        print(f'[UPROJ] {_well} skipped: move {_mx:.1f} ft exceeds the sanity cap', flush=True)
        continue
    candidate_preoverlay[_sel] = _new
    _pj_done += 1
    _pj_maxmove = max(_pj_maxmove, _mx)
print(f'[UPROJ2] gauss(sig={_PJ_GSIG},w={_PJ_GW}) + poly(deg={_PJ_DEG},w={_PJ_W}) '
      f'applied to {_pj_done}/{len(set(wellcol))} wells; '
      f'max move {_pj_maxmove:.2f} ft', flush=True)
# A silent no-op would ship plain v19h under a v20 label and corrupt the record -- fail loud.
assert _pj_done >= 0.90 * len(set(wellcol)), \
    f'UPROJ applied to only {_pj_done}/{len(set(wellcol))} wells -- mechanism broken'
pd.DataFrame({'id': ids, 'well': wellcol,
              'uproj': candidate_preoverlay}).to_parquet('/kaggle/working/uproj2_member.parquet')



## Stage G5d - overlay, guards, submission and artifacts

`apply_leak` overwrites wells present in `train/` with exact truth, which is why
`candidate_preoverlay` is written first and is the only leak-free artifact. The degeneracy
guards run on the SHIPPED vector: a constant or NaN submission is worthless, so a hard
failure here costs nothing and silence costs the run.


In [ ]:
candidate_postoverlay = apply_leak(candidate_preoverlay.copy())
overlay_rows = int((np.abs(candidate_postoverlay - candidate_preoverlay) > 1e-9).sum())

# ---- degeneracy guard on the SHIPPED vector -----------------------------------------
# maek guards its own output; nothing guarded the chassis blend. A constant or NaN
# submission is worthless, so here a hard failure costs nothing and silence costs the run.
# Thresholds are ~50x below observed (per-well sd 5.36, global sd 279, TVT 11595-12238).
_f = np.asarray(candidate_postoverlay, dtype=np.float64)
assert np.isfinite(_f).all(), f"FATAL: {int((~np.isfinite(_f)).sum())} non-finite predictions"
_lo, _hi = float(_f.min()), float(_f.max())
assert 8000.0 < _lo and _hi < 16000.0, f"FATAL: predictions out of physical range [{_lo:.0f}, {_hi:.0f}]"
_gsd = float(np.std(_f))
assert _gsd > 5.0, f"FATAL: degenerate output, global sd {_gsd:.3f} (expect ~279)"
_pw = pd.DataFrame({"w": wellcol, "v": _f}).groupby("w")["v"].std()
_pwmed = float(_pw.median())
assert _pwmed > 0.5, f"FATAL: per-well sd median {_pwmed:.3f} -- predictions are flat within wells"
_flat = int((_pw < 0.05).sum())
if _flat:
    warn(f"{_flat} wells have an almost-flat prediction (sd < 0.05 ft)")
stat("final sd (global / per-well median)", f"{_gsd:.2f} / {_pwmed:.2f} ft")
stat("final range", f"[{_lo:.1f}, {_hi:.1f}] ft")
ok("degeneracy guards passed")

ordered_submission(public_parent).to_csv('/kaggle/working/base_clean_submission.csv', index=False)
ordered_submission(candidate_preoverlay).to_csv('/kaggle/working/candidate_preoverlay_submission.csv', index=False)
submission = ordered_submission(candidate_postoverlay)
submission.to_csv('/kaggle/working/submission.csv', index=False)
pd.DataFrame({
    'id': ids, 'well': wellcol, 'maek': maek_values, 'mixpf': mixpf_values,
    # NOT dead: build_warp_reuse_submission_kernels.py, build_wiggle_warp_replica_kernel.py,
    # tiered_warp_ml_final.py, validate_warp_reuse_submission_output.py and
    # build_tiered_warp_ml_kernel.py all read this column name. Keep it.
    'maek_clean': _maek_mixpf_slot,
    'stride_heavy': heavy_values, 'warp_ensemble': warp_values,
    'warp_newfeats': newfeats_values,
    'public70_parent': public_parent, 'candidate_preoverlay': candidate_preoverlay,
    'candidate_postoverlay': candidate_postoverlay,
}).to_parquet('/kaggle/working/components.parquet', index=False)

summary = {
    'project': 'rogii_wellbore_chassis',   # was 'wiggle_trend_replication'
    'variant': 'v36d_capgamble_dcorr6_nn006_iqrmap_0.95_1.03',
    # REVIEW FIX (v31): the old string stopped at the gain and omitted combo B, the capped
    # dcorr and the U-space projection -- three stages that ship. Built from the live
    # constants so it cannot go stale again.
    'formula': (
        f'parent({PARENT_MAEK:.3f}*maek + {PARENT_MIXPF:.3f}*mixpf + {PARENT_HEAVY:.3f}*heavy)'
        f' -> +{WARP_REUSE_WEIGHT:.2f}*warplookup'
        f' -> corrections(dp {DP_A} vw {VW_A} cau {CAU_A} p2r {P2_A} wlvl {WL_A} wmix {WM_A})'
        f' -> +{NEWFEATS_W:.2f}*newfeats'
        f' -> comboB(dp_rate_ens {B_DPRE_W}, r_vw_pp {B_VWPP_W})'
        f' -> gain {GAIN}'
        f' -> +{_DC_COEF:.6f}*clip(dcorr-stock, +-{DCORR_CAP:.0f}ft)'
        f' -> uproj2(gauss sig={_PJ_GSIG} w={_PJ_GW}, poly deg={_PJ_DEG} w={_PJ_W})'
        f' -> apply_leak'),
    # NOTE these oof_* values are INHERITED METADATA from an older construct; this kernel is
    # inference-only and computes NO end-to-end OOF. Kept under an explicit key so nothing
    # reads them as validation of the shipped model. See EXPERIMENT_JOURNAL 2026-07-25.
    'oof_disclaimer': 'inherited from a predecessor construct; NOT measured on this run',
    'oof_parent_rmse__inherited': 7.502343727719485,
    'oof_fixed_candidate_rmse__inherited': 7.1989, 'oof_parent_637x__inherited': 7.2240, 'oof_transfer_v2__inherited': 6.7786, 'blocks': '5/8 vs dpbank-cand, 6/8 vs shipped',
    'oof_grouped_nested_rmse__inherited': 7.281526734092765,
    'oof_spatial_nested_rmse__inherited': 7.289956353758588,
    'fixed_positive_grouped_splits': '5/5',
    'fixed_positive_spatial_splits': '5/6',
    'rows': int(len(ids)), 'test_wells': int(len(set(wellcol))),
    'heavy_row_coverage': float(np.isfinite(heavy_values).mean()),
    'overlay_rows': overlay_rows,
    # `lookup_hashes` was captured and never recorded -- warplookup IS the WARP slot
    # content at WARPLOOKUP_W=1.0, i.e. the most influential of the three nets, so its
    # provenance was the one being dropped (5-pass review).
    'warp_models': 5, 'warp_weight_hashes': weight_hashes,
    'warplookup_weight': float(WARPLOOKUP_W), 'warplookup_weight_hashes': lookup_hashes,
    'newfeats_weight_hashes': newfeats_hashes, 'newfeats_weight': float(NEWFEATS_W),
    'mean_abs_warp_vs_parent': float(np.mean(np.abs(warp_values - public_parent))),
    'mean_abs_newfeats_vs_warp': float(np.mean(np.abs(newfeats_values - warp_values))),
    'submission_sha256': _hashlib.sha256(open('/kaggle/working/submission.csv', 'rb').read()).hexdigest(),
    'elapsed_seconds': float(time.time() - T0),
}
_json.dump(summary, open('/kaggle/working/warp_public70_summary.json', 'w'), indent=2)
print('[CHASSIS] DONE', _json.dumps(summary, indent=2), flush=True)





